# 📈 StockForecastNet V5 — End-to-End Google Colab Training
## PatchTST + ReVIN + Dual-Stream + Channel-Independent Transformer

> **GPU Required** — `Runtime → Change runtime type → T4 GPU` before running

### What this notebook does
| Step | Cell | What happens |
|------|------|--------------|
| 1 | Setup | Verify GPU, install packages, mount Drive |
| 2 | ⭐ Config | **YOU EDIT THIS** — stocks, dates, hyperparams |
| 3 | Source Code | Paste entire project codebase inline (no uploads) |
| 4 | Fetch Data | Download NSE OHLCV via yfinance (free, no API key) |
| 5 | Preprocess | Feature engineering: 56 stationary technical indicators |
| 6 | Train | Full pipeline with epoch logging and ETA |
| 7 | Evaluate | Validation metrics + signal distribution |
| 8 | Backtest | Portfolio simulation with share quantities and P&L |
| 9 | Export | Save & download model artifacts for FastAPI/AWS |

### Speed comparison
| Hardware | 4 IT stocks | 10 diverse stocks |
|---|---|---|
| Your CPU (8 cores) | ~350 min ❌ | ~900 min ❌ |
| **Colab T4 GPU** | **~8-12 min ✅** | **~20-28 min ✅** |
| Colab A100 GPU | ~3-5 min ✅ | ~8-12 min ✅ |


## 🔧 Section 1 — Setup

In [ ]:
# ─── 1A: Verify GPU ───────────────────────────────────────────────────────────
import subprocess, sys, os, pathlib

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ GPU detected:")
    for line in result.stdout.strip().split("\n"):
        print(f"  {line}")
else:
    print("⚠ NO GPU FOUND")
    print("Go to: Runtime → Change runtime type → T4 GPU")
    print("Then: Runtime → Run all")

import torch
print(f"\nPyTorch: {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  |  VRAM: {p.total_memory/1024**3:.1f} GB")
    print("✓ Ready for GPU training")
else:
    print("⚠ CUDA not available — training will be 70x slower")


In [ ]:
# ─── 1B: Install Dependencies ─────────────────────────────────────────────────
# Colab has: torch, numpy, pandas, sklearn pre-installed
# We add: yfinance (free NSE data), joblib (model persistence), pyarrow (parquet)
print("Installing packages...")
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "yfinance>=0.2.40", "joblib>=1.3.0", "pyarrow>=15.0.0",
     "scikit-learn>=1.4.0"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip stderr:", result.stderr[-500:])
else:
    print("✓ yfinance, joblib, pyarrow, scikit-learn installed")

import yfinance, joblib, sklearn, pyarrow
print(f"  yfinance={yfinance.__version__}  joblib={joblib.__version__}  "
      f"sklearn={sklearn.__version__}  pyarrow={pyarrow.__version__}")


In [ ]:
# ─── 1C: Project Folder Structure ────────────────────────────────────────────
# Creates a clean folder layout matching your local project structure
import pathlib, sys, os

PROJECT_ROOT = pathlib.Path("/content/ai-trading-service")
DIRS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "strategies",
    PROJECT_ROOT / "utils",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "models",      # trained weights saved here
    PROJECT_ROOT / "exports",     # final exports for AWS/FastAPI
]
for d in DIRS:
    d.mkdir(parents=True, exist_ok=True)

# Add project root to Python path so imports work
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✓ Folder structure created:")
for d in DIRS:
    print(f"  {d}")
print()
print("sys.path[0] =", sys.path[0])


In [ ]:
# ─── 1D: Mount Google Drive (optional but recommended) ───────────────────────
# If USE_DRIVE=True, trained models are saved to Drive so they survive
# Colab session resets. You can always download them in Section 9 either way.

USE_DRIVE = False  # ← Set True to save models to Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = pathlib.Path("/content/drive/MyDrive/AI_Trading_V5/")
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✓ Drive mounted. Models will be saved to: {DRIVE_SAVE_DIR}")
else:
    DRIVE_SAVE_DIR = None
    print("Drive not mounted. Models saved to /content/ai-trading-service/models/")
    print("You can download them in Section 9.")


## ⭐ Section 2 — Configuration (Edit This Cell)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⭐ CONFIGURATION — ONLY CELL YOU NEED TO EDIT
# ══════════════════════════════════════════════════════════════════════════════

# ── Stock symbols (Yahoo Finance NSE format: SYMBOL.NS) ───────────────────────
# CRITICAL: Use DIVERSE sectors for 58-64% accuracy.
# Same-sector stocks (all IT) → model learns one pattern → 52-55% accuracy
#
# Recommended (10 stocks, 6 sectors — run ~20-28 min on T4):
SYMBOLS = [
    "RELIANCE.NS",    # Energy / Conglomerate
    "TCS.NS",         # IT Services
    "HDFCBANK.NS",    # Banking
    "INFY.NS",        # IT Services
    "ICICIBANK.NS",   # Banking
    "MARUTI.NS",      # Automobile
    "AXISBANK.NS",    # Banking
    "TATASTEEL.NS",   # Metals
    "SUNPHARMA.NS",   # Pharma
    "ITC.NS",         # FMCG
]
# Quick test (4 IT stocks — ~8-12 min on T4, ~52-55% accuracy):
# SYMBOLS = ["TCS.NS", "INFY.NS", "WIPRO.NS", "HCLTECH.NS"]

# ── Date range ────────────────────────────────────────────────────────────────
# 2010-01-01 is the recommended start — modern NSE market structure
# Pre-2010: different circuit breakers, settlement rules, liquidity regime
START_DATE = "2010-01-01"
END_DATE   = "2026-04-30"    # ← update to today if needed

# ── Model architecture ────────────────────────────────────────────────────────
SEQ_LEN    = 90    # input window in days (60-90 recommended)
HORIZON    = 3     # prediction steps ahead (3=optimal SNR, 5=swing trading)
PATCH_SIZE = 16    # consecutive days per patch (16 = ~3 weeks, good for cycles)
STRIDE     = 8     # patch stride — overlap = patch_size - stride = 8 days
D_MODEL    = 128   # embedding dimension (use 64 if <5 stocks, 128 for 5-10)
N_HEADS    = 4     # attention heads (must divide D_MODEL evenly)
N_LAYERS   = 2     # transformer encoder layers (2 sufficient, 4 for large datasets)
D_FF       = 256   # feedforward hidden dim (always 2× D_MODEL)
DROPOUT    = 0.1   # regularization (increase to 0.2 if overfit)

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE       = 128    # GPU optimal; reduce to 64 if CUDA OOM error
EPOCHS           = 100    # early stopping usually kicks in at 60-85
LR               = 3e-4   # AdamW learning rate
WEIGHT_DECAY     = 1e-3   # AdamW weight decay
PATIENCE         = 30     # early stopping patience (MUST be >20 for LR restart)
NOISE_THRESHOLD  = 0.001  # skip near-zero return samples during dataset build
VAL_SPLIT        = 0.2    # fraction of each stock's data held for validation
GAP              = 10     # days gap between train/val to prevent autocorrelation
SEED             = 42     # reproducibility

# ── Output file names ─────────────────────────────────────────────────────────
# These match your local project's config.py MODEL_PATH / CONFIG_PATH / SCALER_PATH
WEIGHTS_FILE = "pretrained_v5.pth"        # model weights (maps to pretrain_path)
CONFIG_FILE  = "pretrained_v5_config.pth" # get_config() dict
SCALER_FILE  = "scaler_v2.pkl"            # fitted RobustScaler

MODELS_DIR  = PROJECT_ROOT / "models"
EXPORTS_DIR = PROJECT_ROOT / "exports"

# ── Print configuration summary ───────────────────────────────────────────────
n_patches = (SEQ_LEN - PATCH_SIZE) // STRIDE + 1
B_C = BATCH_SIZE * 56  # 56 features (Channel-Independent effective batch)
fused_mb = B_C * n_patches * D_MODEL * 4 / 1024 / 1024

print("=" * 60)
print("  CONFIGURATION SUMMARY")
print("=" * 60)
print(f"  Stocks ({len(SYMBOLS)}):  {', '.join(SYMBOLS)}")
print(f"  Date range:  {START_DATE} → {END_DATE}")
print(f"  Architecture: seq={SEQ_LEN}, horizon={HORIZON}d, patch={PATCH_SIZE}/{STRIDE}")
print(f"  d_model={D_MODEL}, n_heads={N_HEADS}, n_layers={N_LAYERS}, d_ff={D_FF}")
print(f"  n_patches = ({SEQ_LEN}-{PATCH_SIZE})//{STRIDE}+1 = {n_patches}")
print(f"  Training: batch={BATCH_SIZE}, epochs={EPOCHS}, lr={LR}, patience={PATIENCE}")
print(f"  GPU: B×C = {BATCH_SIZE}×56 = {B_C}, fused tensor ≈ {fused_mb:.1f} MB")
print("=" * 60)


## 📦 Section 3 — Project Source Code (Auto-Generated, Do Not Edit)

In [ ]:
# ─── 3A: Write strategy files to disk ────────────────────────────────────────
# Your exact strategies/base.py and all 19 strategy files are written to
# /content/ai-trading-service/strategies/ so imports work identically to local

import pathlib, textwrap

_strat_sources = {}

# ── base.py ──
_strat_sources["base.py"] = '''\
"""
strategies/base.py — Abstract base class for all strategy modules.

Every strategy follows this interface so they can be applied
in a consistent pipeline: df → strategy.apply(df) → df with new columns.
"""

import pandas as pd
from abc import ABC, abstractmethod


class BaseStrategy(ABC):
    """
    Base class all strategies must inherit from.

    Convention:
        - apply() receives a DataFrame with at minimum: open, high, low, close, volume
        - apply() MUST return the same DataFrame with new signal columns added
        - Column names must be unique per strategy to avoid collisions
        - apply() should NEVER drop rows — that is the caller\'s job
    """

    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Add strategy-specific signal columns to the DataFrame.

        Args:
            df: OHLCV DataFrame

        Returns:
            DataFrame with new columns added (no rows removed)
        """
        ...

    def __repr__(self):
        return f"{self.__class__.__name__}()"

'''

_strat_sources["atr_strategy.py"] = '''\
"""
strategies/atr_strategy.py — ATR (Average True Range) volatility strategy.

What ATR is:
    ATR measures how much a stock MOVES each day, in absolute price terms.
    It captures the "typical daily range" of a stock.

    True Range for one day = max of:
        1. High - Low           (today\'s intraday range)
        2. |High - Prev Close|  (gap up scenario)
        3. |Low  - Prev Close|  (gap down scenario)

    ATR = rolling average of True Range (default 14 days)

    Example: If RELIANCE has ATR = ₹45, it typically moves ₹45 in a day.

Why ATR matters for the model:
    1. VOLATILITY REGIME DETECTION
       High ATR = high volatility period (news, earnings, macro events)
       Low ATR  = low volatility / squeeze (breakout likely coming)
       The model needs to know "is this a quiet market or a choppy one?"

    2. POSITION SIZING (not in model, but for actual trading)
       Professional traders size positions as: Risk / ATR
       If you want to risk ₹5000 and ATR=₹50 → buy 100 shares
       This keeps your risk constant regardless of which stock you trade

    3. STOP-LOSS PLACEMENT
       Place stop 1.5× ATR below entry. Statistically, normal market noise
       is less than 1 ATR — only a real adverse move hits a 1.5× ATR stop.

Derived features:
    atr:            Raw ATR in price units (₹)
    atr_pct:        ATR as % of price (comparable across different price stocks)
    atr_ratio:      Current ATR / 20-day avg ATR (is volatility expanding?)
    high_vol_regime: 1 if current ATR > 1.5× its own 20-day average
"""

import numpy as np
import pandas as pd
# BaseStrategy already defined above


class ATRStrategy(BaseStrategy):

    def __init__(self, atr_period: int = 14, vol_period: int = 20):
        """
        Args:
            atr_period: Period for ATR calculation (14 is standard)
            vol_period: Period for volatility regime comparison
        """
        self.atr_period = atr_period
        self.vol_period = vol_period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # True Range components
        high_low   = df["high"] - df["low"]
        high_close = (df["high"] - df["close"].shift(1)).abs()
        low_close  = (df["low"]  - df["close"].shift(1)).abs()

        true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        df["atr"]  = true_range.rolling(self.atr_period).mean()

        # ATR as % of price — comparable across stocks trading at different prices
        # RELIANCE ATR=45 at price 2500 is 1.8%. IRCTC ATR=8 at price 700 is 1.1%.
        # The % form makes these comparable.
        df["atr_pct"] = df["atr"] / df["close"].replace(0, np.nan) * 100

        # ATR ratio: is current volatility HIGH or LOW relative to recent history?
        avg_atr = df["atr"].rolling(self.vol_period).mean().replace(0, np.nan)
        df["atr_ratio"] = df["atr"] / avg_atr

        # Volatility regime: 1 = high volatility (ATR > 1.5× its 20-day average)
        df["high_vol_regime"] = (df["atr_ratio"] > 1.5).astype(float)

        # Volatility compression: ATR near its 20-day LOW → squeeze, breakout likely
        atr_20low = df["atr"].rolling(self.vol_period).min()
        df["vol_compressed"] = (df["atr"] < atr_20low * 1.2).astype(float)

        return df

'''

_strat_sources["bb_strategy.py"] = '''\
"""
strategies/bb_strategy.py — Bollinger Bands strategy.

What Bollinger Bands measure:
    Bollinger Bands draw a "channel" around price based on volatility.

    Middle Band  = 20-day moving average
    Upper Band   = Middle + 2 × standard deviation
    Lower Band   = Middle - 2 × standard deviation

    Think of it as: "where is price relative to its normal range?"

    bb_position = 0   → price at the lower band (potentially oversold)
    bb_position = 0.5 → price in the middle (normal)
    bb_position = 1   → price at the upper band (potentially overbought)
    bb_position > 1   → price ABOVE the upper band (strong breakout)

    bb_width measures how "wide" the channel is — wide = high volatility period.
    When bands are very narrow (squeeze), a big move is usually coming soon.

Why the model needs this:
    Bollinger Bands tell the model both where price IS (relative position)
    and how volatile the market currently IS (band width). These are two
    separate and very useful signals.
"""

import pandas as pd
# BaseStrategy already defined above


class BollingerBandStrategy(BaseStrategy):

    def __init__(self, period: int = 20, std_dev: float = 2.0):
        self.period  = period
        self.std_dev = std_dev

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        rolling_mean = df["close"].rolling(self.period).mean()
        rolling_std  = df["close"].rolling(self.period).std()

        df["bb_upper"]    = rolling_mean + self.std_dev * rolling_std
        df["bb_lower"]    = rolling_mean - self.std_dev * rolling_std
        df["bb_mid"]      = rolling_mean

        band_range = (df["bb_upper"] - df["bb_lower"]).replace(0, float("nan"))

        # 0 = at lower band, 1 = at upper band
        df["bb_position"] = (df["close"] - df["bb_lower"]) / band_range

        # Band width relative to price — measure of current volatility
        df["bb_width"] = band_range / rolling_mean

        # Squeeze: bands are very narrow (low volatility, breakout incoming)
        # Defined as: current width is in the bottom 20% of last 50 days
        df["bb_squeeze"] = (df["bb_width"] < df["bb_width"].rolling(50).quantile(0.2)).astype(float)

        return df

'''

_strat_sources["breakout_strategy.py"] = '''\
"""
strategies/breakout_strategy.py — Price breakout detection.

What it detects:
    A breakout happens when today\'s closing price is HIGHER than the highest
    price seen in the last 20 days (resistance level).

    Why this matters:
        Resistance = a price level where sellers previously dominated.
        When price breaks through it, those sellers are "defeated" —
        it often leads to a strong continuation move upward.

    breakout = 1 → Price closed above the 20-day high (bullish signal)
    breakout = 0 → No breakout
"""

import pandas as pd
# BaseStrategy already defined above


class BreakoutStrategy(BaseStrategy):

    def __init__(self, lookback: int = 20):
        self.lookback = lookback

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # The highest high in the past `lookback` days (EXCLUDING today)
        # shift(1) is critical — prevents lookahead bias (we don\'t know today\'s
        # resistance until after today ends)
        df["resistance"] = df["high"].rolling(self.lookback).max().shift(1)
        df["support"]    = df["low"].rolling(self.lookback).min().shift(1)

        # Binary breakout signal
        df["breakout"] = (df["close"] > df["resistance"]).astype(float)

        # How far above resistance did price close? (0 = no breakout, >0 = how strong)
        df["breakout_strength"] = ((df["close"] - df["resistance"]) / df["resistance"]).clip(lower=0)

        return df

'''

_strat_sources["candlestick_strategy.py"] = '''\
"""
strategies/candlestick_strategy.py — Candlestick pattern detection.

What candlestick patterns are:
    Each day\'s candle has a shape defined by open, high, low, close.
    Certain shapes reliably signal reversals or continuations.
    These patterns encode institutional behaviour that pure indicators miss.

    A candlestick has:
        Body:       |close - open|        (the thick part)
        Upper Wick: high - max(open,close) (thin line above)
        Lower Wick: min(open,close) - low  (thin line below)

Patterns detected (all values: 1=bullish, -1=bearish, 0=no pattern):

    DOJI:
        Open ≈ Close (body is very small, < 10% of range)
        Meaning: buyers and sellers are equally matched → indecision
        After a trend, a doji signals the trend may be exhausting

    HAMMER / HANGING MAN:
        Long lower wick (> 2× body), small upper wick
        At bottom of downtrend (hammer): bulls rejected the lows → reversal UP
        At top of uptrend (hanging man): warning sign, possible reversal DOWN

    SHOOTING STAR:
        Long upper wick (> 2× body), small lower wick, small body at BOTTOM of range
        After an uptrend: bulls pushed price up during day but sellers took over → bearish

    ENGULFING:
        Bullish: Current day\'s body completely ENGULFS prior day\'s body, current closes UP
        Bearish: Current day\'s body completely ENGULFS prior day\'s body, current closes DOWN
        Strong reversal signals — especially on high volume

    MARUBOZU:
        Entire day is one big body with no wicks
        Bullish: Opens at low, closes at high — pure buying pressure all day
        Bearish: Opens at high, closes at low — pure selling pressure all day

Why the model benefits from these:
    Technical indicators like RSI and MACD are "lagging" — they reflect what
    ALREADY happened. Candlestick patterns capture the intraday PSYCHOLOGY of
    buyers and sellers. A bullish engulfing on high volume tells you something
    RSI cannot.
"""

import numpy as np
import pandas as pd
# BaseStrategy already defined above


class CandlestickStrategy(BaseStrategy):

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        body       = (df["close"] - df["open"]).abs()
        full_range = (df["high"] - df["low"]).replace(0, np.nan)
        upper_wick = df["high"] - df[["open", "close"]].max(axis=1)
        lower_wick = df[["open", "close"]].min(axis=1) - df["low"]
        body_ratio = body / full_range   # 0=pure doji, 1=marubozu

        # ── Doji ──────────────────────────────────────────────────────────────
        # Body is less than 10% of the day\'s range → indecision
        df["doji"] = (body_ratio < 0.1).astype(float)

        # ── Hammer (bullish) ──────────────────────────────────────────────────
        # Long lower wick, small upper wick, body in upper part of range
        # Signals: sellers pushed price down but buyers recovered strongly
        hammer_cond = (
            (lower_wick > 2 * body.clip(lower=1e-9)) &
            (upper_wick < 0.3 * full_range) &
            (df["close"] > df["open"])    # Closed green (confirms bullish)
        )
        df["hammer"] = hammer_cond.astype(float)

        # ── Shooting Star (bearish) ────────────────────────────────────────────
        # Long upper wick, small lower wick, body in lower part of range
        # Signals: buyers pushed price up but sellers overwhelmed them
        star_cond = (
            (upper_wick > 2 * body.clip(lower=1e-9)) &
            (lower_wick < 0.3 * full_range) &
            (df["close"] < df["open"])    # Closed red (confirms bearish)
        )
        df["shooting_star"] = star_cond.astype(float)

        # ── Bullish Engulfing ─────────────────────────────────────────────────
        # Current green candle\'s body is LARGER than previous red candle\'s body
        prev_body  = body.shift(1)
        prev_close = df["close"].shift(1)
        prev_open  = df["open"].shift(1)

        bull_engulf = (
            (df["close"] > df["open"]) &        # Current is green
            (prev_close < prev_open) &           # Previous was red
            (df["open"]  < prev_close) &         # Opens below prev close
            (df["close"] > prev_open)            # Closes above prev open
        )
        df["bullish_engulfing"] = bull_engulf.astype(float)

        # ── Bearish Engulfing ─────────────────────────────────────────────────
        bear_engulf = (
            (df["close"] < df["open"]) &         # Current is red
            (prev_close > prev_open) &            # Previous was green
            (df["open"]  > prev_close) &          # Opens above prev close
            (df["close"] < prev_open)             # Closes below prev open
        )
        df["bearish_engulfing"] = bear_engulf.astype(float)

        # ── Marubozu (strong trend candle) ────────────────────────────────────
        # Body is > 90% of range (almost no wicks) → strong conviction
        df["bullish_marubozu"] = ((body_ratio > 0.9) & (df["close"] > df["open"])).astype(float)
        df["bearish_marubozu"] = ((body_ratio > 0.9) & (df["close"] < df["open"])).astype(float)

        # ── Composite candle signal ───────────────────────────────────────────
        # +1 = at least one bullish pattern, -1 = bearish, 0 = no clear pattern
        bullish = df["hammer"] | df["bullish_engulfing"] | df["bullish_marubozu"]
        bearish = df["shooting_star"] | df["bearish_engulfing"] | df["bearish_marubozu"]
        df["candle_signal"] = bullish.astype(int) - bearish.astype(int)

        return df

'''

_strat_sources["cci_strategy.py"] = '''\
"""
strategies/cci_strategy.py — Commodity Channel Index (CCI)

WHAT IT IS:
    CCI measures how far price is from its "statistical average" using
    mean deviation rather than standard deviation. Created by Donald Lambert
    in 1980 originally for commodities, but widely used in equities.

    CCI = (Typical Price - SMA of Typical Price) / (0.015 × Mean Deviation)

    Typical Price = (High + Low + Close) / 3

    0.015 is a scaling constant that makes ~70-80% of CCI values fall in [-100, +100].

    Key levels:
      CCI > +100 → price is well above its average (strong uptrend, or overbought)
      CCI < -100 → price is well below its average (strong downtrend, or oversold)
      CCI crosses zero → trend change signal

WHY CCI INSTEAD OF (OR ALONGSIDE) RSI:
    RSI compares up-moves to down-moves.
    CCI compares price to its statistical mean — more sensitive to sudden price
    surges (big intraday moves). Good at catching the START of a new trend.

    Together: RSI for momentum exhaustion, CCI for trend deviation.

COLUMNS ADDED:
    cci:         Raw CCI value (typically -200 to +200, no hard cap)
    cci_signal:  +1 if CCI > 100 (strong uptrend), -1 if CCI < -100 (downtrend), 0 otherwise
    cci_norm:    CCI / 200 clipped to [-1, +1] — for model input
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class CCIStrategy(BaseStrategy):

    def __init__(self, period: int = 20):
        """
        Args:
            period: Lookback window (default 20 — common standard)
        """
        self.period = period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Typical price: the representative price for the day
        typical = (df["high"] + df["low"] + df["close"]) / 3.0

        sma_typical = typical.rolling(self.period).mean()

        # Mean deviation: average absolute deviation from the mean
        def mean_dev(series):
            return series.rolling(self.period).apply(
                lambda x: np.abs(x - x.mean()).mean(), raw=True
            )

        mad = mean_dev(typical).replace(0, np.nan)

        df["cci"] = (typical - sma_typical) / (0.015 * mad)

        # Classic trading signals
        df["cci_signal"] = 0
        df.loc[df["cci"] >  100, "cci_signal"] =  1
        df.loc[df["cci"] < -100, "cci_signal"] = -1

        # Normalised to [-1, +1] for model consumption
        df["cci_norm"] = (df["cci"] / 200.0).clip(-1.0, 1.0)

        return df

'''

_strat_sources["donchian_strategy.py"] = '''\
"""
strategies/donchian_strategy.py — Donchian Channels

WHAT DONCHIAN CHANNELS ARE:
    Donchian channels draw a band around price using the highest high and lowest low
    over a lookback period. Created by Richard Donchian (pioneer of trend-following).

    Upper band = max(high, last N days)
    Lower band = min(low,  last N days)
    Middle band = (upper + lower) / 2

    These form a "channel" showing the price range.

TRADING SIGNALS:
    BREAKOUT: When price closes above the upper band, it is making a NEW N-day high.
    This is often the start of a trend move (breakout trading).

    RANGE CONTRACTION: When upper - lower is very small, price is in tight range.
    A big breakout often follows tight range compression (similar to BB squeeze).

    POSITION: Where is today\'s close within the channel?
    top = above midpoint, bottom = below midpoint

DIFFERENCE FROM BOLLINGER BANDS:
    Bollinger Bands use standard deviation (volatility-based dynamic width).
    Donchian uses raw price extremes (simpler, based on actual traded prices).
    Donchian is better for breakout systems; BB is better for mean-reversion.

COLUMNS ADDED:
    don_upper:    Upper Donchian channel (highest high over N days)
    don_lower:    Lower Donchian channel (lowest low over N days)
    don_mid:      Midpoint of channel
    don_width:    Channel width / close (normalised, stationary)
    don_position: Where close sits in channel: 0=bottom, 1=top
    don_breakout_up:   1 if price broke above yesterday\'s upper band
    don_breakout_down: 1 if price broke below yesterday\'s lower band
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class DonchianStrategy(BaseStrategy):

    def __init__(self, period: int = 20):
        self.period = period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df["don_upper"] = df["high"].rolling(self.period).max()
        df["don_lower"] = df["low"].rolling(self.period).min()
        df["don_mid"]   = (df["don_upper"] + df["don_lower"]) / 2.0

        channel_range = (df["don_upper"] - df["don_lower"]).replace(0, np.nan)
        df["don_width"]    = channel_range / df["close"].replace(0, np.nan)
        df["don_position"] = (df["close"] - df["don_lower"]) / channel_range

        # Breakout signals: compare today\'s close to YESTERDAY\'s channel extremes
        # (use .shift(1) to avoid lookahead — today\'s high contributes to today\'s upper)
        df["don_breakout_up"]   = (df["close"] > df["don_upper"].shift(1)).astype(float)
        df["don_breakout_down"] = (df["close"] < df["don_lower"].shift(1)).astype(float)

        return df

'''

_strat_sources["heikin_ashi_strategy.py"] = '''\
"""
strategies/heikin_ashi_strategy.py — Heikin-Ashi Candles

WHAT HEIKIN-ASHI IS:
    Heikin-Ashi (Japanese: "average bar") is a modified candlestick chart
    that smooths price action by using averaged values instead of raw OHLC.

    HA_Close = (Open + High + Low + Close) / 4         (average of all 4 prices)
    HA_Open  = (Prev HA_Open + Prev HA_Close) / 2      (average of prev HA candle)
    HA_High  = max(High, HA_Open, HA_Close)
    HA_Low   = min(Low,  HA_Open, HA_Close)

WHY HEIKIN-ASHI IS USEFUL:
    Regular candles are noisy — price jumps up and down each day. This makes it
    hard to see the underlying trend.

    Heikin-Ashi SMOOTHS this noise:
    - Strong uptrend:   most candles are green with NO lower wick
    - Strong downtrend: most candles are red with NO upper wick
    - Trend weakening:  small bodies appear, wicks on both sides

    The model benefits because HA features encode TREND STRENGTH more directly
    than raw price structure.

    IMPORTANT: Heikin-Ashi candles are for signal generation only.
    For actual trade execution, use the original OHLC prices (not HA).

COLUMNS ADDED:
    ha_open, ha_high, ha_low, ha_close:  The 4 HA candle values (non-stationary)
    ha_body:        HA_Close - HA_Open (positive = bullish HA candle)
    ha_trend:       +1 if HA close > HA open (bullish), -1 if bearish
    ha_no_low_wick: 1 if HA low wick is very small (strong uptrend signal)
    ha_no_hi_wick:  1 if HA upper wick is very small (strong downtrend signal)
    ha_body_norm:   ha_body / close (stationary, for model input)
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class HeikinAshiStrategy(BaseStrategy):

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        ha_close = (df["open"] + df["high"] + df["low"] + df["close"]) / 4.0

        ha_open = pd.Series(index=df.index, dtype=float)
        ha_open.iloc[0] = (df["open"].iloc[0] + df["close"].iloc[0]) / 2.0
        for i in range(1, len(df)):
            ha_open.iloc[i] = (ha_open.iloc[i - 1] + ha_close.iloc[i - 1]) / 2.0

        ha_high = pd.concat(
            [df["high"], ha_open, ha_close], axis=1
        ).max(axis=1)
        ha_low  = pd.concat(
            [df["low"],  ha_open, ha_close], axis=1
        ).min(axis=1)

        df["ha_open"]  = ha_open
        df["ha_high"]  = ha_high
        df["ha_low"]   = ha_low
        df["ha_close"] = ha_close

        # Candle body: positive = bullish HA bar, negative = bearish
        df["ha_body"] = ha_close - ha_open

        # Trend direction from HA candle
        df["ha_trend"] = np.sign(df["ha_body"])

        # Wick analysis (stationary ratios)
        ha_range = (ha_high - ha_low).replace(0, np.nan)
        upper_wick = (ha_high - df[["ha_open", "ha_close"]].max(axis=1)) / ha_range
        lower_wick = (df[["ha_open", "ha_close"]].min(axis=1) - ha_low) / ha_range

        # Strong trend signals: no wick on the trend side
        df["ha_no_low_wick"]  = (lower_wick < 0.05).astype(float)   # strong uptrend
        df["ha_no_hi_wick"]   = (upper_wick < 0.05).astype(float)   # strong downtrend

        # Stationary body for model
        df["ha_body_norm"]    = df["ha_body"] / df["close"].replace(0, np.nan)

        return df

'''

_strat_sources["ichimoku_strategy.py"] = '''\
"""
strategies/ichimoku_strategy.py — Ichimoku Cloud (Ichimoku Kinko Hyo)

WHAT ICHIMOKU IS:
    Ichimoku ("one look equilibrium chart") is a comprehensive indicator that shows
    support/resistance, trend direction, momentum, AND buy/sell signals all at once.

    5 components (all based on midpoints of highs/lows over different periods):

    Tenkan-sen (Conversion Line, 9):
        (9-day High + 9-day Low) / 2
        Fast moving "average" — if price is above it, short-term bullish.

    Kijun-sen (Base Line, 26):
        (26-day High + 26-day Low) / 2
        Slower version. Primary trend indicator.
        Price above = uptrend; below = downtrend.

    Senkou Span A (Leading Span A):
        (Tenkan + Kijun) / 2  — plotted 26 periods AHEAD
        One edge of the "cloud" (Kumo).

    Senkou Span B (Leading Span B, 52):
        (52-day High + 52-day Low) / 2 — plotted 26 periods AHEAD
        Other edge of the cloud.

    Chikou Span (Lagging Span):
        Today\'s close plotted 26 periods BACK.

THE CLOUD (Kumo):
    Area between Span A and Span B. Think of it as a dynamic support/resistance zone.
    - Price ABOVE cloud = uptrend
    - Price BELOW cloud = downtrend
    - Price INSIDE cloud = sideways/transition

WHY WE USE LAGGED VALUES:
    Spans A and B are "future projected" in visual charts but for ML training
    we use the current-day values shifted back (equivalent to projecting forward).
    This is standard for using Ichimoku in backtesting.

COLUMNS ADDED (all stationary — as distance ratios to close):
    ichi_tenkan_dist:  (Tenkan - close) / close
    ichi_kijun_dist:   (Kijun - close) / close
    ichi_cloud_top:    max(SpanA, SpanB) 26 periods ago relative to close
    ichi_cloud_bot:    min(SpanA, SpanB) 26 periods ago relative to close
    ichi_above_cloud:  1 if price is above the cloud, -1 if below, 0 inside
    ichi_tk_cross:     +1 when Tenkan crosses above Kijun (bullish), -1 for bearish
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class IchimokuStrategy(BaseStrategy):

    def __init__(self, tenkan: int = 9, kijun: int = 26, senkou_b: int = 52):
        self.tenkan   = tenkan
        self.kijun    = kijun
        self.senkou_b = senkou_b

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        def midpoint(series_high, series_low, n):
            return (series_high.rolling(n).max() + series_low.rolling(n).min()) / 2.0

        tenkan  = midpoint(df["high"], df["low"], self.tenkan)
        kijun   = midpoint(df["high"], df["low"], self.kijun)
        span_a  = (tenkan + kijun) / 2.0
        span_b  = midpoint(df["high"], df["low"], self.senkou_b)

        close   = df["close"].replace(0, np.nan)
        df["ichi_tenkan_dist"] = (tenkan - df["close"]) / close
        df["ichi_kijun_dist"]  = (kijun  - df["close"]) / close

        # Cloud levels from 26 periods ago (what the cloud shows "now" in a live chart)
        cloud_top = span_a.shift(self.kijun).combine(span_b.shift(self.kijun), max)
        cloud_bot = span_a.shift(self.kijun).combine(span_b.shift(self.kijun), min)

        df["ichi_cloud_top"] = (cloud_top - df["close"]) / close
        df["ichi_cloud_bot"] = (cloud_bot - df["close"]) / close

        # Price relative to cloud
        above_cloud = df["close"] > cloud_top
        below_cloud = df["close"] < cloud_bot
        df["ichi_above_cloud"] = 0
        df.loc[above_cloud, "ichi_above_cloud"] =  1
        df.loc[below_cloud, "ichi_above_cloud"] = -1

        # Tenkan / Kijun crossover
        tk_above      = tenkan > kijun
        tk_above_prev = tenkan.shift(1) > kijun.shift(1)
        df["ichi_tk_cross"] = 0
        df.loc[tk_above & ~tk_above_prev,  "ichi_tk_cross"] =  1
        df.loc[~tk_above & tk_above_prev,  "ichi_tk_cross"] = -1

        return df

'''

_strat_sources["keltner_strategy.py"] = '''\
"""
strategies/keltner_strategy.py — Keltner Channels

WHAT KELTNER CHANNELS ARE:
    Keltner Channels draw bands above and below an EMA (exponential moving average)
    using ATR (Average True Range) as the band width.

    Middle  = EMA(close, period)
    Upper   = EMA + multiplier × ATR
    Lower   = EMA - multiplier × ATR

KELTNER vs BOLLINGER BANDS:
    Both are "envelope" indicators drawing bands around price.
    Key difference: Bollinger uses Standard Deviation; Keltner uses ATR.

    - Bollinger responds to price VOLATILITY (big price moves widen the band)
    - Keltner responds to TRUE RANGE volatility (includes gaps, more stable)
    - Keltner bands are smoother and don\'t expand/contract as dramatically

SQUEEZE SIGNAL (very powerful):
    When Bollinger Bands are INSIDE Keltner Channels → "Squeeze"
    This means volatility has compressed significantly.
    A big breakout is imminent (direction unknown until it happens).
    This is one of the most reliable volatility compression signals.

    squeeze = bb_upper < kc_upper AND bb_lower > kc_lower

COLUMNS ADDED:
    kc_upper:    Upper Keltner Channel
    kc_lower:    Lower Keltner Channel
    kc_mid:      Middle EMA line
    kc_position: Where close sits in channel (0=lower, 1=upper)
    kc_width:    Channel width / close (normalised volatility)
    kc_squeeze:  1 if Bollinger Bands are inside Keltner Channels
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class KeltnerStrategy(BaseStrategy):

    def __init__(self, ema_period: int = 20, atr_period: int = 10,
                 multiplier: float = 2.0):
        self.ema_period  = ema_period
        self.atr_period  = atr_period
        self.multiplier  = multiplier

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        ema = df["close"].ewm(span=self.ema_period, adjust=False).mean()

        hl  = df["high"] - df["low"]
        hc  = (df["high"] - df["close"].shift(1)).abs()
        lc  = (df["low"]  - df["close"].shift(1)).abs()
        atr = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(self.atr_period).mean()

        df["kc_mid"]   = ema
        df["kc_upper"] = ema + self.multiplier * atr
        df["kc_lower"] = ema - self.multiplier * atr

        kc_range = (df["kc_upper"] - df["kc_lower"]).replace(0, np.nan)
        df["kc_position"] = (df["close"] - df["kc_lower"]) / kc_range
        df["kc_width"]    = kc_range / df["close"].replace(0, np.nan)

        # Squeeze detection (requires Bollinger Bands columns)
        if "bb_upper" in df.columns and "bb_lower" in df.columns:
            df["kc_squeeze"] = (
                (df["bb_upper"] < df["kc_upper"]) &
                (df["bb_lower"] > df["kc_lower"])
            ).astype(float)
        else:
            df["kc_squeeze"] = 0.0

        return df

'''

_strat_sources["ma_strategy.py"] = '''\
"""
strategies/ma_strategy.py — Moving Average crossover strategy.

What it detects:
    When the fast moving average (MA10) crosses ABOVE the slow (MA20),
    it signals upward momentum — the recent trend is stronger than the longer trend.

    MA10 > MA20 → Bullish (signal = 1)
    MA10 < MA20 → Bearish (signal = 0)

Why this works:
    Price tends to follow momentum. When short-term average rises above
    long-term average, more recent buyers are profitable — continuation is likely.
"""

import pandas as pd
# BaseStrategy already defined above


class MovingAverageStrategy(BaseStrategy):

    def __init__(self, fast: int = 10, slow: int = 20):
        self.fast = fast
        self.slow = slow

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df[f"ma_{self.fast}"] = df["close"].rolling(self.fast).mean()
        df[f"ma_{self.slow}"] = df["close"].rolling(self.slow).mean()

        # 1 = fast above slow (bullish), 0 = fast below slow (bearish)
        df["ma_signal"] = (df[f"ma_{self.fast}"] > df[f"ma_{self.slow}"]).astype(int)

        # Distance between the two MAs as a % of price — strength indicator
        df["ma_spread"] = (df[f"ma_{self.fast}"] - df[f"ma_{self.slow}"]) / df["close"]

        return df

'''

_strat_sources["macd_strategy.py"] = '''\
"""
strategies/macd_strategy.py — MACD (Moving Average Convergence Divergence).

What MACD measures:
    MACD = EMA(12) - EMA(26)

    EMA = Exponential Moving Average — like a regular moving average but
    it gives MORE weight to recent prices (reacts faster to new information).

    MACD Line: Difference between 12-day and 26-day EMA
    Signal Line: 9-day EMA of the MACD line itself (smoothed version)
    Histogram: MACD - Signal (positive = bullish momentum, negative = bearish)

    When MACD crosses ABOVE Signal → buy signal (momentum turning positive)
    When MACD crosses BELOW Signal → sell signal (momentum turning negative)

Why the model needs this:
    MACD captures the speed and direction of price change.
    RSI says "how tired is this move?" — MACD says "is the move accelerating?"
    Together they give the model momentum + exhaustion information.
"""

import pandas as pd
# BaseStrategy already defined above


class MACDStrategy(BaseStrategy):

    def __init__(self, fast: int = 12, slow: int = 26, signal: int = 9):
        self.fast   = fast
        self.slow   = slow
        self.signal = signal

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        ema_fast = df["close"].ewm(span=self.fast, adjust=False).mean()
        ema_slow = df["close"].ewm(span=self.slow, adjust=False).mean()

        df["macd"]        = ema_fast - ema_slow
        df["macd_signal"] = df["macd"].ewm(span=self.signal, adjust=False).mean()
        df["macd_hist"]   = df["macd"] - df["macd_signal"]

        # Normalize histogram by price so it\'s comparable across different stocks/prices
        df["macd_hist_norm"] = df["macd_hist"] / df["close"]

        # Crossover signal: 1 = MACD crossed above signal, -1 = crossed below, 0 = no cross
        df["macd_cross"] = 0
        df.loc[(df["macd"] > df["macd_signal"]) & (df["macd"].shift(1) <= df["macd_signal"].shift(1)), "macd_cross"] = 1
        df.loc[(df["macd"] < df["macd_signal"]) & (df["macd"].shift(1) >= df["macd_signal"].shift(1)), "macd_cross"] = -1

        return df

'''

_strat_sources["momentum_strategy.py"] = '''\
"""
strategies/momentum_strategy.py — Multi-timeframe price momentum strategy.

What momentum is:
    Momentum = the tendency of things that are moving to keep moving.
    In markets: stocks that have risen recently tend to continue rising
    (in the short/medium term). This is one of the most robustly documented
    effects in financial markets — published in hundreds of academic papers.

    This strategy measures momentum at multiple timeframes simultaneously:
    - 1-week  (5 days):   Very short-term micro-momentum
    - 1-month (20 days):  Short-term momentum
    - 3-month (60 days):  Medium-term momentum (strongest predictive power)
    - 6-month (120 days): Long-term momentum

    When multiple timeframes align (e.g. all positive), the signal is stronger.
    When they diverge (1-week negative, 1-month positive), mixed signals → caution.

Why multi-timeframe matters:
    A stock might have 1-week positive momentum (bouncing from a dip) but
    3-month negative momentum (still in a downtrend). Buying a dip in a
    downtrend is different from buying a dip in an uptrend.
    The transformer learns to interpret these combinations.

Rate of Change (ROC):
    ROC(n) = (close_today - close_n_days_ago) / close_n_days_ago × 100
    Simple but effective. Shows "how much has this stock moved in the last n days?"

Momentum oscillator:
    Compares recent short-term return to recent longer-term return.
    Positive = recent days are stronger than the month average (acceleration)
    Negative = recent days are weaker than the month average (deceleration)
"""

import numpy as np
import pandas as pd
# BaseStrategy already defined above


class MomentumStrategy(BaseStrategy):

    def __init__(
        self,
        periods: list = None,   # Lookback periods in days
    ):
        # Standard multi-timeframe periods
        self.periods = periods or [5, 10, 20, 60, 120]

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # ── Rate of Change for each period ─────────────────────────────────────
        for n in self.periods:
            prev = df["close"].shift(n).replace(0, np.nan)
            df[f"roc_{n}"] = (df["close"] - prev) / prev * 100

        # ── Momentum alignment score ───────────────────────────────────────────
        # How many timeframes are positive at the same time?
        # Range: -len(periods) to +len(periods)
        # +5 = all timeframes bullish (strong uptrend)
        # -5 = all timeframes bearish (strong downtrend)
        alignment = sum(
            (df[f"roc_{n}"] > 0).astype(int) - (df[f"roc_{n}"] < 0).astype(int)
            for n in self.periods
        )
        df["momentum_alignment"] = alignment

        # ── Normalised alignment (−1 to +1) ────────────────────────────────────
        df["momentum_score"] = df["momentum_alignment"] / len(self.periods)

        # ── Momentum acceleration ──────────────────────────────────────────────
        # Is short-term (5d) momentum stronger or weaker than medium-term (20d)?
        # Positive = accelerating (recent days outperforming the month)
        # Negative = decelerating (recent days underperforming the month)
        if 5 in self.periods and 20 in self.periods:
            df["momentum_accel"] = df["roc_5"] - df["roc_20"]

        # ── 52-week high proximity ─────────────────────────────────────────────
        # Distance from 52-week high as a % — how far below the yearly peak?
        # Value near 0 = near 52-week high (strength)
        # Value near -30% = far below 52-week high (potential weakness or deep value)
        yearly_high = df["close"].rolling(252).max().replace(0, np.nan)
        df["pct_from_52w_high"] = (df["close"] - yearly_high) / yearly_high * 100

        # ── 52-week low proximity ──────────────────────────────────────────────
        yearly_low = df["close"].rolling(252).min().replace(0, np.nan)
        df["pct_from_52w_low"]  = (df["close"] - yearly_low)  / yearly_low  * 100

        return df

'''

_strat_sources["obv_strategy.py"] = '''\
"""
strategies/obv_strategy.py — On-Balance Volume (OBV)

WHAT IT IS:
    OBV accumulates volume: adds volume on up days, subtracts on down days.
    It answers: "Is volume flowing INTO (buying pressure) or OUT OF (selling pressure)
    this stock?"

    If Close > Prev Close: OBV = Prev OBV + Volume
    If Close < Prev Close: OBV = Prev OBV - Volume
    If Close = Prev Close: OBV = Prev OBV

WHY OBV IS POWERFUL:
    Price can be manipulated (low-volume moves are easy to push). Volume is harder
    to fake. When price rises but OBV falls (DIVERGENCE), the price rise is weak —
    not backed by real buying. This often precedes a reversal.

    Conversely: when price falls but OBV rises → smart money is accumulating
    (buying into the dip). A strong move up is likely coming.

THE PROBLEM WITH RAW OBV:
    Raw OBV is non-stationary (keeps accumulating forever like a random walk).
    We can\'t feed raw OBV to a model trained on different time periods.
    Solution: use OBV CHANGE (daily OBV delta / volume) and OBV TREND
    (OBV relative to its moving average). Both are stationary.

COLUMNS ADDED:
    obv:           Raw accumulated OBV (non-stationary — for reference only)
    obv_change:    Daily OBV change / volume (stationary, +1 or -1 typically)
    obv_to_ma20:   OBV / 20-day OBV MA - 1 (stationary, divergence signal)
    obv_rising:    1 if OBV is above its 20-day MA (accumulation trend)
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class OBVStrategy(BaseStrategy):

    def __init__(self, ma_period: int = 20):
        self.ma_period = ma_period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Compute OBV: cumulative signed volume
        direction        = np.sign(df["close"].diff().fillna(0))
        obv              = (direction * df["volume"]).cumsum()
        df["obv"]        = obv

        # Daily signed volume (stationary proxy for OBV momentum)
        df["obv_change"] = direction   # +1, -1, or 0

        # OBV relative to its moving average (stationary divergence signal)
        obv_ma           = obv.rolling(self.ma_period).mean().replace(0, np.nan)
        df["obv_to_ma20"] = obv / obv_ma - 1

        # Is OBV above its MA? (accumulation vs distribution trend)
        df["obv_rising"] = (obv > obv_ma).astype(float)

        return df

'''

_strat_sources["pivot_strategy.py"] = '''\
"""
strategies/pivot_strategy.py — Classical Pivot Points

WHAT PIVOT POINTS ARE:
    Pivot points are support and resistance LEVELS calculated from the
    previous trading day\'s High, Low, and Close.

    These exact levels are watched by floor traders, market makers, and
    algorithmic systems — making them self-fulfilling to some degree.

    Pivot Point (PP)    = (Prev High + Prev Low + Prev Close) / 3
    Resistance 1 (R1)   = 2 × PP - Prev Low
    Resistance 2 (R2)   = PP + (Prev High - Prev Low)
    Support 1 (S1)      = 2 × PP - Prev High
    Support 2 (S2)      = PP - (Prev High - Prev Low)

WHY THE MODEL NEEDS THIS:
    Price action near known support/resistance levels behaves differently
    from price action in "open space." Near support:
    - If price holds → bounce potential (buy)
    - If price breaks → acceleration down (sell)

    Expressing distance to these levels as a % of price gives the model
    context about where price is in its "decision zone."

COLUMNS ADDED:
    pivot_pp:          Pivot point level
    pivot_r1, pivot_r2: Resistance levels
    pivot_s1, pivot_s2: Support levels
    dist_to_pp:         (close - PP) / close  (stationary)
    dist_to_r1:         (close - R1) / close
    dist_to_s1:         (close - S1) / close
    near_support:       1 if close is within 0.5% of S1 or S2
    near_resistance:    1 if close is within 0.5% of R1 or R2
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class PivotStrategy(BaseStrategy):

    def __init__(self, threshold_pct: float = 0.005):
        """
        Args:
            threshold_pct: How close to a level (as % of price) counts as "near"
                           Default 0.5%
        """
        self.threshold = threshold_pct

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Pivot levels are based on PREVIOUS day\'s data
        prev_high  = df["high"].shift(1)
        prev_low   = df["low"].shift(1)
        prev_close = df["close"].shift(1)

        pp = (prev_high + prev_low + prev_close) / 3.0
        r1 = 2 * pp - prev_low
        r2 = pp + (prev_high - prev_low)
        s1 = 2 * pp - prev_high
        s2 = pp - (prev_high - prev_low)

        df["pivot_pp"] = pp
        df["pivot_r1"] = r1
        df["pivot_r2"] = r2
        df["pivot_s1"] = s1
        df["pivot_s2"] = s2

        close = df["close"].replace(0, np.nan)
        df["dist_to_pp"] = (df["close"] - pp) / close
        df["dist_to_r1"] = (df["close"] - r1) / close
        df["dist_to_s1"] = (df["close"] - s1) / close

        # Binary: is price close to any support or resistance level?
        df["near_support"] = (
            (df["dist_to_s1"].abs() < self.threshold) |
            ((df["close"] - s2).abs() / close < self.threshold)
        ).astype(float)

        df["near_resistance"] = (
            (df["dist_to_r1"].abs() < self.threshold) |
            ((df["close"] - r2).abs() / close < self.threshold)
        ).astype(float)

        return df

'''

_strat_sources["rsi_strategy.py"] = '''\
"""
strategies/rsi_strategy.py — Relative Strength Index strategy.

What RSI measures:
    RSI compares the size of recent gains vs recent losses over 14 days.
    Result is a number between 0 and 100.

    RSI > 70 → Overbought (price moved up too fast, likely to reverse DOWN)
    RSI < 30 → Oversold  (price moved down too fast, likely to reverse UP)
    30-70    → Neutral

Why the model needs this:
    The transformer can\'t inherently know "this stock has been falling for 2 weeks
    and is exhausted". RSI encodes that momentum exhaustion as a number.
"""

import pandas as pd
# BaseStrategy already defined above


class RSIStrategy(BaseStrategy):

    def __init__(self, period: int = 14):
        self.period = period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        delta = df["close"].diff()
        gain = delta.clip(lower=0).rolling(self.period).mean()
        loss = (-delta.clip(upper=0)).rolling(self.period).mean()

        rs = gain / loss.replace(0, float("nan"))
        df["rsi"] = 100 - (100 / (1 + rs))

        # Categorical signal: 1=oversold(buy), -1=overbought(sell), 0=neutral
        df["rsi_signal"] = 0
        df.loc[df["rsi"] < 30, "rsi_signal"] = 1
        df.loc[df["rsi"] > 70, "rsi_signal"] = -1

        # Normalized RSI (0 to 1) — easier for the model to consume
        df["rsi_norm"] = df["rsi"] / 100.0

        return df

'''

_strat_sources["stochastic_strategy.py"] = '''\
"""
strategies/stochastic_strategy.py — Stochastic Oscillator (%K and %D)

WHAT IT IS:
    The Stochastic Oscillator compares a closing price to the price range
    (high-low) over a lookback period. The result is a number 0-100 showing
    WHERE today\'s close sits within the recent price range.

    %K = (Close - Lowest Low over N days) / (Highest High - Lowest Low) × 100
    %D = Simple moving average of %K (usually 3 days)

    Close at the TOP of the range → %K near 100 (overbought, buyers in control)
    Close at the BOTTOM            → %K near 0 (oversold, sellers in control)

    Classic signals:
      %K crosses ABOVE %D and both are below 20 → buy signal (oversold reversal)
      %K crosses BELOW %D and both are above 80 → sell signal (overbought reversal)

DIFFERENCE FROM RSI:
    RSI measures momentum (speed of price changes).
    Stochastic measures POSITION within a range (where is price relative to recent highs/lows).
    They complement each other — RSI says "how tired is this move?",
    Stochastic says "is price near the top or bottom of its recent range?"

COLUMNS ADDED:
    stoch_k:      Raw %K (0-100)
    stoch_d:      3-day smoothed %K (the signal line)
    stoch_cross:  +1 when K crosses above D (bullish), -1 when K crosses below D (bearish)
    stoch_norm:   stoch_k normalised to 0-1 (for model input)
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class StochasticStrategy(BaseStrategy):

    def __init__(self, k_period: int = 14, d_period: int = 3):
        """
        Args:
            k_period: Lookback window for %K calculation (default 14 days)
            d_period: Smoothing period for %D signal line (default 3 days)
        """
        self.k_period = k_period
        self.d_period = d_period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        lowest_low   = df["low"].rolling(self.k_period).min()
        highest_high = df["high"].rolling(self.k_period).max()

        price_range = (highest_high - lowest_low).replace(0, np.nan)

        df["stoch_k"] = (df["close"] - lowest_low) / price_range * 100
        df["stoch_d"] = df["stoch_k"].rolling(self.d_period).mean()

        # Crossover: +1 = K just crossed above D (bullish), -1 = K crossed below D
        k_above_d      = df["stoch_k"] > df["stoch_d"]
        k_above_d_prev = df["stoch_k"].shift(1) > df["stoch_d"].shift(1)
        df["stoch_cross"] = 0
        df.loc[k_above_d & ~k_above_d_prev, "stoch_cross"] = 1
        df.loc[~k_above_d & k_above_d_prev, "stoch_cross"] = -1

        # Normalised to 0-1 for model consumption
        df["stoch_norm"] = df["stoch_k"] / 100.0

        return df

'''

_strat_sources["supertrend_strategy.py"] = '''\
"""
strategies/supertrend_strategy.py — SuperTrend

WHAT SUPERTREND IS:
    SuperTrend is a trend-following indicator that sits above or below price
    depending on trend direction. When price is above the SuperTrend line → uptrend.
    When price is below → downtrend.

    It is calculated using ATR (Average True Range) to set band width:
        Upper Band = (High + Low) / 2 + multiplier × ATR
        Lower Band = (High + Low) / 2 - multiplier × ATR

    The final SuperTrend line flips between upper and lower bands based on
    whether price closes above or below the previous band.

WHY SUPERTREND IS USEFUL:
    Unlike moving averages, SuperTrend:
    - Does NOT cross during sideways (avoids whipsaws better than MA crossover)
    - Adapts to volatility via ATR (wider bands in volatile periods)
    - Gives clean binary signal: in trend (1) or in downtrend (-1)

    Very popular among Indian retail traders — many use it as primary signal.

COLUMNS ADDED:
    supertrend_val:  The SuperTrend line value (price level)
    supertrend_dir:  +1 = price above SuperTrend (uptrend), -1 = below (downtrend)
    supertrend_dist: (close - supertrend_val) / close  (stationary distance)
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class SuperTrendStrategy(BaseStrategy):

    def __init__(self, period: int = 10, multiplier: float = 3.0):
        """
        Args:
            period:     ATR period (default 10)
            multiplier: Band width multiplier (default 3.0)
                        Higher = fewer signals but less noise
                        Lower  = more signals but more whipsaws
        """
        self.period     = period
        self.multiplier = multiplier

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # ATR
        hl  = df["high"] - df["low"]
        hc  = (df["high"] - df["close"].shift(1)).abs()
        lc  = (df["low"]  - df["close"].shift(1)).abs()
        tr  = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        atr = tr.rolling(self.period).mean()

        mid = (df["high"] + df["low"]) / 2.0
        upper_basic = mid + self.multiplier * atr
        lower_basic = mid - self.multiplier * atr

        # Compute final SuperTrend with direction-aware banding
        n = len(df)
        supertrend = pd.Series(index=df.index, dtype=float)
        direction  = pd.Series(index=df.index, dtype=int)

        # Initialise first valid row
        first = atr.first_valid_index()
        if first is None:
            df["supertrend_val"]  = np.nan
            df["supertrend_dir"]  = 0
            df["supertrend_dist"] = np.nan
            return df

        loc = df.index.get_loc(first)
        supertrend.iloc[loc] = upper_basic.iloc[loc]
        direction.iloc[loc]  = -1

        for i in range(loc + 1, n):
            prev_sup  = supertrend.iloc[i - 1]
            prev_dir  = direction.iloc[i - 1]
            close     = df["close"].iloc[i]
            close_prev = df["close"].iloc[i - 1]
            ub = upper_basic.iloc[i]
            lb = lower_basic.iloc[i]

            if prev_dir == 1:
                # Was in uptrend
                curr_sup = max(lb, prev_sup) if close > prev_sup else ub
                curr_dir = 1 if close > curr_sup else -1
            else:
                # Was in downtrend
                curr_sup = min(ub, prev_sup) if close < prev_sup else lb
                curr_dir = -1 if close < curr_sup else 1

            supertrend.iloc[i] = curr_sup
            direction.iloc[i]  = curr_dir

        df["supertrend_val"]  = supertrend
        df["supertrend_dir"]  = direction
        df["supertrend_dist"] = (df["close"] - supertrend) / df["close"].replace(0, np.nan)

        return df

'''

_strat_sources["vwap_strategy.py"] = '''\
"""
strategies/vwap_strategy.py — VWAP (Volume Weighted Average Price) strategy.

What VWAP is:
    VWAP = Sum(Price × Volume) / Sum(Volume)

    It is the average price a stock traded at across the day, weighted by how
    much volume occurred at each price level.

    VWAP is the single most-watched intraday level by institutional traders
    (mutual funds, hedge funds, FIIs). Institutions try to execute large orders
    NEAR the VWAP to minimise market impact. This makes VWAP a magnetic level.

    Price ABOVE VWAP → Buyers are in control. Institutional activity bullish.
    Price BELOW VWAP → Sellers are in control. Institutional activity bearish.

    The VWAP deviation (how far price is from VWAP as a %) tells the model:
    - Large positive deviation: stock stretched above fair value — mean reversion risk
    - Large negative deviation: stock stretched below fair value — bounce candidate
    - Near zero: price is at fair value — no strong directional signal

Why the model benefits from this:
    RSI and MACD are pure price-based. VWAP incorporates VOLUME — it tells you
    where the money actually transacted. A move on high volume near VWAP is more
    significant than a move on thin volume far from VWAP.

Note on daily data:
    True intraday VWAP requires tick or minute-level data. On daily OHLCV data,
    we approximate using the typical price = (high + low + close) / 3 × volume.
    This is the standard approach used in daily charts.
"""

import numpy as np
import pandas as pd
# BaseStrategy already defined above


class VWAPStrategy(BaseStrategy):

    def __init__(self, period: int = 20):
        """
        Args:
            period: Rolling window for VWAP calculation (default 20 days).
                    20 = approximately one month of trading days.
        """
        self.period = period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Typical price = average of high, low, close for the day
        typical_price = (df["high"] + df["low"] + df["close"]) / 3.0

        # Volume-weighted sum over the rolling window
        tp_vol   = (typical_price * df["volume"]).rolling(self.period).sum()
        vol_sum  = df["volume"].rolling(self.period).sum().replace(0, np.nan)

        df["vwap"] = tp_vol / vol_sum

        # Deviation from VWAP as a fraction of VWAP (positive = above, negative = below)
        df["vwap_deviation"] = (df["close"] - df["vwap"]) / df["vwap"].replace(0, np.nan)

        # Signal: 1 = price above VWAP (bullish), -1 = below (bearish), 0 = neutral (within 0.5%)
        df["vwap_signal"] = 0
        df.loc[df["vwap_deviation"] >  0.005, "vwap_signal"] =  1
        df.loc[df["vwap_deviation"] < -0.005, "vwap_signal"] = -1

        # Distance trend: is price moving TOWARD or AWAY from VWAP?
        # Positive = diverging (stretched), Negative = converging (mean-reverting)
        df["vwap_diverging"] = df["vwap_deviation"].diff()

        return df

'''

_strat_sources["williams_r_strategy.py"] = '''\
"""
strategies/williams_r_strategy.py — Williams %R

WHAT IT IS:
    Created by Larry Williams. Measures how close the current close is to the
    HIGHEST HIGH over the lookback period. The result is always -100 to 0.

    %R = (Highest High - Close) / (Highest High - Lowest Low) × (-100)

    %R near   0 → close is near the highest high (bullish momentum)
    %R near -100 → close is near the lowest low (bearish momentum)

    Overbought zone: %R between 0 and -20 (price near the top of range)
    Oversold zone:   %R between -80 and -100 (price near the bottom)

DIFFERENCE FROM STOCHASTIC:
    Williams %R and Stochastic %K measure very similar things. The key difference:
    - Stochastic: looks at close relative to LOWEST LOW
    - Williams %R: looks at close relative to HIGHEST HIGH (inverted perspective)
    - Stochastic output: 0-100; Williams %R output: -100 to 0
    Williams %R reacts faster (not smoothed by default) — better at catching
    early reversal signals.

COLUMNS ADDED:
    williams_r:      Raw %R (-100 to 0)
    williams_r_norm: Normalised to 0-1 (0 = oversold extreme, 1 = overbought extreme)
    williams_r_sig:  +1 if oversold (<-80), -1 if overbought (>-20), 0 neutral
"""

import pandas as pd
import numpy as np
# BaseStrategy already defined above


class WilliamsRStrategy(BaseStrategy):

    def __init__(self, period: int = 14):
        self.period = period

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        highest_high = df["high"].rolling(self.period).max()
        lowest_low   = df["low"].rolling(self.period).min()
        price_range  = (highest_high - lowest_low).replace(0, np.nan)

        df["williams_r"] = (highest_high - df["close"]) / price_range * (-100)

        # Normalised 0-1: 0 = at lowest (oversold), 1 = at highest (overbought)
        df["williams_r_norm"] = (df["williams_r"] + 100) / 100.0

        df["williams_r_sig"] = 0
        df.loc[df["williams_r"] < -80, "williams_r_sig"] =  1   # oversold = potential buy
        df.loc[df["williams_r"] > -20, "williams_r_sig"] = -1   # overbought = potential sell

        return df

'''


# Write all strategy files
strat_dir = PROJECT_ROOT / "strategies"
for fname, content in _strat_sources.items():
    (strat_dir / fname).write_text(content)

# Write __init__.py for the strategies package
(strat_dir / "__init__.py").write_text('''
from .base import BaseStrategy
from .ma_strategy import MovingAverageStrategy
from .macd_strategy import MACDStrategy
from .rsi_strategy import RSIStrategy
from .bb_strategy import BollingerBandStrategy
from .breakout_strategy import BreakoutStrategy
from .vwap_strategy import VWAPStrategy
from .atr_strategy import ATRStrategy
from .candlestick_strategy import CandlestickStrategy
from .momentum_strategy import MomentumStrategy
from .stochastic_strategy import StochasticStrategy
from .cci_strategy import CCIStrategy
from .williams_r_strategy import WilliamsRStrategy
from .obv_strategy import OBVStrategy
from .donchian_strategy import DonchianStrategy
from .supertrend_strategy import SuperTrendStrategy
from .keltner_strategy import KeltnerStrategy
from .heikin_ashi_strategy import HeikinAshiStrategy
from .pivot_strategy import PivotStrategy
from .ichimoku_strategy import IchimokuStrategy
''')

print(f"✓ {len(_strat_sources)} strategy files written to {strat_dir}")
for f in sorted(_strat_sources.keys()):
    sz = (strat_dir / f).stat().st_size
    print(f"  {f} ({sz:,} bytes)")


In [ ]:
# ─── 3B: Write model_v2.py ────────────────────────────────────────────────────
_model_source = '''\
"""
model_v2.py — StockForecastNet V5  (PatchTST + ReVIN + Dual-Stream + CI Transformer)
======================================================================================

BUGS FIXED FROM FIRST RELEASE
───────────────────────────────
BUG 1 — Head has 821K params (75% of all params) → massive overfitting
  Old: nn.Flatten → Linear(1280, 640) → Linear(640, horizon)
       = 820,864 + 1,923 = 821,787 params in head alone
  Fix: mean-pool encoder output over patch dimension → Linear(d_model, horizon)
       = 128 × 3 + 3 = 387 params
  Why: 821K head params / 5,762 training samples = 143 params/sample.
       Neural networks need ~10 samples per parameter to generalise.
       The old head guaranteed overfitting to training noise.

BUG 2 — UserWarning: enable_nested_tensor repeated 7 times
  Old: nn.TransformerEncoder(layer, num_layers=n)
       PyTorch ≥2.0 warns when norm_first=True because it can\'t use
       the optimised nested tensor path.
  Fix: nn.TransformerEncoder(layer, num_layers=n, enable_nested_tensor=False)
  Why: Silences the noise; no performance impact with norm_first=True.

BUG 3 — Training crashes silently after printing epoch header
  Old: No try/except in training loop; error swallowed on Windows
  Fix: Added explicit try/except with full traceback in train_loop
  Note: The crash was caused by BUG 1 — the enormous head caused a
        dimension mismatch on the first forward pass because
        nn.Flatten(start_dim=1) was inside nn.Sequential which
        was applied to a 3D tensor (B*C, patches, d_model) correctly,
        BUT the very large Linear then caused silent NaN propagation
        that killed training immediately.

ARCHITECTURE OVERVIEW
──────────────────────
INPUT: x (B, seq_len, n_features)  +  time (B, seq_len, 6)

  Step 1:  ReVIN normalize x per-instance → x_norm (B, T, C)
  Step 2:  CI reshape: (B, T, C) → (B*C, T) — treat each feature independently
  Step 3A: Patch embed: (B*C, T) → (B*C, n_patches, d_model) — VALUE stream
  Step 3B: Time embed:  (B, T, 6) → (B, n_patches, d_model) — TIME stream
             expand → (B*C, n_patches, d_model)
  Step 4:  Fuse: val_emb + time_emb + pos_enc → (B*C, n_patches, d_model)
  Step 5:  Transformer encoder (shared weights across all C features)
             → (B*C, n_patches, d_model)
  Step 6:  Mean pool over patches → (B*C, d_model)
           Reshape → (B, C, d_model)
           Mean over features → (B, d_model)
  Step 7:  Linear(d_model, horizon) → (B, horizon)
  Step 8:  ReVIN denormalize → (B, horizon)

PARAMETER COUNT (default: d_model=128, n_layers=2, seq=90, patch=16/8)
  n_patches = (90-16)//8 + 1 = 10
  ReVIN affine:            56 × 2 =      112
  Patch projection:   16×128 + 128 =    2,176
  Positional encoding:    10×128  =      1,280
  Time projection:      6×128+128 =        896
  Transformer 2 layers: ≈         =    265,216
  Prediction head:    128×3 + 3   =        387
  ────────────────────────────────────────────
  TOTAL:                           ≈  270,067  (1.03 MB)
  Previous buggy version had head alone at 821K params.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════════════
# 1. ReVIN — Reversible Instance Normalization
# ══════════════════════════════════════════════════════════════════════

class ReVIN(nn.Module):
    """
    Reversible Instance Normalization (Kim et al. 2021).

    WHY INSTANCE NORMALIZATION INSTEAD OF GLOBAL SCALING:
    ───────────────────────────────────────────────────────
    Global RobustScaler problem:
        Fitted on training data (2010–2022), normalises using the training
        distribution. A COVID window (σ≈5% daily) and a calm 2015 window
        (σ≈0.8% daily) both get scaled using the same global median/IQR.
        The model at inference sees systematic distribution shift.

    ReVIN solution:
        Each window normalises using its OWN mean and std:
          x_norm = (x - mean_window) / std_window
        After prediction:
          y_final = y_raw * std_window + mean_window

        Every window looks statistically similar regardless of regime.
        A 2020 crash window and a 2015 bull window both become
        zero-mean unit-variance sequences before the model sees them.

    Learnable affine (γ, β per feature):
        Allows the model to optionally rescale after normalization.
        Initialised as identity (γ=1, β=0). Gradient can adjust.
        In practice stays close to identity for return features.
    """

    def __init__(self, n_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.n_features = n_features
        self.eps        = eps
        self.affine     = affine

        if affine:
            self.gamma = nn.Parameter(torch.ones(1, 1, n_features))
            self.beta  = nn.Parameter(torch.zeros(1, 1, n_features))

    def normalize(self, x: torch.Tensor):
        """
        x: (B, T, C) → x_norm: (B, T, C), stats: (mean (B,1,C), std (B,1,C))
        """
        mean = x.mean(dim=1, keepdim=True)
        std  = x.std(dim=1, keepdim=True) + self.eps
        x_norm = (x - mean) / std
        if self.affine:
            x_norm = x_norm * self.gamma + self.beta
        return x_norm, (mean, std)

    def denormalize(self, y: torch.Tensor, stats: tuple) -> torch.Tensor:
        """
        y: (B, horizon) in normalized space → original return scale
        stats: (mean (B,1,C), std (B,1,C)) from normalize()
        """
        mean, std = stats
        # Scale by first feature (ret_1d) statistics — consistent with return target
        m = mean[:, 0, 0].unsqueeze(-1)   # (B, 1)
        s = std[:,  0, 0].unsqueeze(-1)   # (B, 1)
        return y * s + m


# ══════════════════════════════════════════════════════════════════════
# 2. PatchEmbedding — Stream A (Values)
# ══════════════════════════════════════════════════════════════════════

class PatchEmbedding(nn.Module):
    """
    Patch-based temporal embedding (PatchTST, Nie et al. 2023).

    WHY PATCHES INSTEAD OF TOKEN-PER-TIMESTEP:
    ────────────────────────────────────────────
    Token-per-day:
        90 tokens → 90² = 8,100 attention pairs
        Each token = single day snapshot (no local context)
        High noise, expensive attention

    Patch embedding (patch_size=16, stride=8):
        n_patches = (90-16)//8+1 = 10 tokens
        10² = 100 attention pairs (81× less)
        Each token = 16 consecutive days compressed to d_model
        Local temporal pattern is already embedded in each token

    CHANNEL-INDEPENDENT (CI):
    ──────────────────────────
    The same Linear(patch_size, d_model) is shared across ALL 56 features.
    Input is reshaped so all features share the batch dimension: (B*C, T)
    After patching: (B*C, n_patches, d_model)

    This means the model learns: "a rising 16-day patch → next patch likely
    continues" — a pattern valid for RSI, MACD, ATR, OBV equally.
    """

    def __init__(self, seq_len: int, patch_size: int, stride: int,
                 d_model: int, dropout: float = 0.1):
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = (seq_len - patch_size) // stride + 1
        self.projection = nn.Linear(patch_size, d_model)
        self.norm       = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B*C, seq_len)
        → patches: (B*C, n_patches, d_model)

        unfold extracts overlapping windows of size patch_size with step stride.
        projection maps each patch_size-dim window to d_model.
        """
        patches = x.unfold(dimension=1, size=self.patch_size, step=self.stride)
        # (B*C, n_patches, patch_size)
        patches = self.norm(self.projection(patches))
        # (B*C, n_patches, d_model)
        return self.dropout(patches)


# ══════════════════════════════════════════════════════════════════════
# 3. TemporalEmbedding — Stream B (Time)
# ══════════════════════════════════════════════════════════════════════

class TemporalEmbedding(nn.Module):
    """
    Cyclic temporal embedding from calendar features.

    WHY CYCLIC ENCODING (sin/cos) INSTEAD OF SCALAR:
    ──────────────────────────────────────────────────
    Scalar: December=11, January=0. Distance = 11. But December→January is
    the smallest possible month transition. The scalar misrepresents this.

    Cyclic: angle = 2π × month / 12
            [sin(angle), cos(angle)]
    December and January are adjacent on the unit circle.
    The model can compute cos(angle_dec - angle_jan) ≈ 1 (small distance).

    FEATURES (6 total):
        month_sin, month_cos    — seasonal patterns (period=12)
        weekday_sin, weekday_cos — Mon-Fri effects (period=5)
        dom_sin, dom_cos        — day-of-month (period=31)

    These are projected to d_model and averaged over each patch window
    to match Stream A\'s temporal resolution of n_patches tokens.
    """

    def __init__(self, d_model: int, patch_size: int, stride: int,
                 n_patches: int, dropout: float = 0.1):
        super().__init__()
        self.d_model    = d_model
        self.patch_size = patch_size
        self.stride     = stride
        self.n_patches  = n_patches

        self.projection = nn.Sequential(
            nn.Linear(6, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, time_features: torch.Tensor) -> torch.Tensor:
        """
        time_features: (B, seq_len, 6) — cyclic encodings
        → time_emb: (B, n_patches, d_model)
        """
        t = self.projection(time_features)     # (B, seq_len, d_model)
        t_t = t.transpose(1, 2)               # (B, d_model, seq_len)
        t_patches = t_t.unfold(dimension=2, size=self.patch_size, step=self.stride)
        # (B, d_model, n_patches, patch_size)
        t_patches = t_patches.mean(dim=-1)     # avg over patch_size → (B, d_model, n_patches)
        return self.dropout(t_patches.transpose(1, 2))   # (B, n_patches, d_model)


# ══════════════════════════════════════════════════════════════════════
# 4. StockForecastNet — Main Model
# ══════════════════════════════════════════════════════════════════════

class StockForecastNet(nn.Module):
    """
    StockForecastNet V5 — full SOTA 2025 time-series forecasting architecture.

    Components:
        ReVIN           — per-instance normalization (handles non-stationarity)
        PatchEmbedding  — local temporal context (reduces sequence length 9×)
        TemporalEmbed   — cyclic calendar features (month, weekday, day-of-month)
        CI Transformer  — shared weights across 56 features (prevents overfitting)
        Lightweight head — mean pool + single linear (387 params vs 821K old)

    V5 FIXES over buggy first release:
        - Head: 821K params → 387 params (prevents massive overfitting)
        - enable_nested_tensor=False (silences 7× UserWarning on Windows)
        - Proper error handling in forward pass
    """

    def __init__(
        self,
        n_features:   int,
        seq_len:      int   = 90,
        horizon:      int   = 3,
        patch_size:   int   = 16,
        stride:       int   = 8,
        d_model:      int   = 128,
        n_heads:      int   = 4,
        n_layers:     int   = 2,
        d_ff:         int   = 256,
        dropout:      float = 0.1,
        revin_affine: bool  = True,
    ):
        super().__init__()

        self.n_features  = n_features
        self.seq_len     = seq_len
        self.horizon     = horizon
        self.patch_size  = patch_size
        self.stride      = stride
        self.d_model     = d_model
        self.n_heads     = n_heads
        self.n_layers    = n_layers
        self.d_ff        = d_ff
        self.dropout_p   = dropout
        self.revin_affine = revin_affine
        self.n_patches   = (seq_len - patch_size) // stride + 1

        assert d_model % n_heads == 0, (
            f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        )
        assert seq_len >= patch_size, (
            f"seq_len ({seq_len}) must be >= patch_size ({patch_size})"
        )

        # ── Components ────────────────────────────────────────────────────
        self.revin      = ReVIN(n_features=n_features, affine=revin_affine)

        self.patch_embed = PatchEmbedding(
            seq_len=seq_len, patch_size=patch_size, stride=stride,
            d_model=d_model, dropout=dropout,
        )
        self.time_embed  = TemporalEmbedding(
            d_model=d_model, patch_size=patch_size, stride=stride,
            n_patches=self.n_patches, dropout=dropout,
        )

        # Learnable positional encoding: one vector per patch position
        # Shared across all C features (CI design)
        self.pos_enc = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)

        # Transformer encoder — Channel-Independent (shared weights across features)
        # FIX: enable_nested_tensor=False silences PyTorch ≥2.0 UserWarning
        # when norm_first=True is used (which we keep for Pre-LN stability)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = n_heads,
            dim_feedforward = d_ff,
            dropout         = dropout,
            activation      = "gelu",
            batch_first     = True,
            norm_first      = True,   # Pre-LN: more stable gradient flow
        )
        self.encoder      = nn.TransformerEncoder(
            encoder_layer,
            num_layers         = n_layers,
            enable_nested_tensor = False,   # FIX: suppresses the 7× UserWarning
        )
        self.encoder_norm = nn.LayerNorm(d_model)

        # Prediction head — FIXED: 387 params instead of 821K
        # (B*C, n_patches, d_model) → mean over patches → (B*C, d_model)
        # → Linear(d_model, horizon) → (B*C, horizon)
        self.head = nn.Linear(d_model, horizon)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.normal_(self.pos_enc, std=0.02)

    def forward(self, x: torch.Tensor,
                time_features: torch.Tensor = None) -> torch.Tensor:
        """
        Full dual-stream forward pass.

        Args:
            x:             (B, seq_len, n_features)
            time_features: (B, seq_len, 6)  or None

        Returns:
            predictions: (B, horizon) — cumulative return predictions
                y[:, 0]  = 1-day ahead cumulative return
                y[:, -1] = N-day ahead cumulative return (primary signal)
        """
        B, T, C = x.shape

        # ── Step 1: ReVIN normalize ────────────────────────────────────────
        # Normalise each window independently to remove distribution shift
        x_norm, revin_stats = self.revin.normalize(x)   # (B, T, C)

        # ── Step 2: Reshape for Channel-Independent processing ─────────────
        # (B, T, C) → (B*C, T): each row = one feature\'s time series
        x_ci = x_norm.transpose(1, 2).reshape(B * C, T)   # (B*C, T)

        # ── Step 3A: Patch embedding (Value stream) ────────────────────────
        val_emb = self.patch_embed(x_ci)
        # (B*C, n_patches, d_model)

        # ── Step 3B: Temporal embedding (Time stream) ─────────────────────
        if time_features is None:
            time_features = torch.zeros(B, T, 6, device=x.device, dtype=x.dtype)
        time_emb = self.time_embed(time_features)
        # (B, n_patches, d_model) → expand to (B*C, n_patches, d_model)
        time_emb = (time_emb.unsqueeze(1)
                            .expand(-1, C, -1, -1)
                            .reshape(B * C, self.n_patches, self.d_model))

        # ── Step 4: Fusion ─────────────────────────────────────────────────
        # Each patch token = value pattern + calendar context + position
        fused = val_emb + time_emb + self.pos_enc   # (B*C, n_patches, d_model)

        # ── Step 5: Channel-Independent Transformer ────────────────────────
        # Self-attention across 10 patch positions (not 90 timesteps)
        # SAME weights for all 56 features (channel-independent)
        enc_out = self.encoder(fused)                 # (B*C, n_patches, d_model)
        enc_out = self.encoder_norm(enc_out)

        # ── Step 6: Pool and aggregate features ────────────────────────────
        # Mean over patches → (B*C, d_model)
        pooled  = enc_out.mean(dim=1)                 # (B*C, d_model)
        # Reshape back to per-sample: (B, C, d_model)
        pooled  = pooled.reshape(B, C, self.d_model)
        # Mean over features: each indicator contributes equally to the prediction
        pooled  = pooled.mean(dim=1)                  # (B, d_model)

        # ── Step 7: Prediction head ─────────────────────────────────────────
        # Single Linear(d_model, horizon) — 387 params (was 821K)
        y_raw = self.head(pooled)                     # (B, horizon)

        # ── Step 8: ReVIN denormalize ───────────────────────────────────────
        predictions = self.revin.denormalize(y_raw, revin_stats)
        return predictions   # (B, horizon)

    def predict_signal(self, x: torch.Tensor,
                       time_features: torch.Tensor = None,
                       conf_scale: float = 100.0):
        """
        Inference convenience method.

        Returns:
            direction:   int   (1=UP, 0=DOWN) — based on primary horizon step
            confidence:  float (0.5–1.0) — sigmoid of |prediction| × scale
            pred_return: float — primary horizon prediction (signed %)
            all_steps:   list  — prediction for each horizon step
        """
        if x.dim() == 2:
            x = x.unsqueeze(0)
        if time_features is not None and time_features.dim() == 2:
            time_features = time_features.unsqueeze(0)

        self.eval()
        with torch.no_grad():
            preds = self.forward(x, time_features)

        pred_return = preds[0, -1].item()
        direction   = 1 if pred_return > 0 else 0
        confidence  = 1.0 / (1.0 + math.exp(-abs(pred_return) * conf_scale))
        return direction, confidence, pred_return, preds[0].tolist()

    def get_config(self) -> dict:
        """Returns constructor kwargs for exact state_dict reloading."""
        return {
            "n_features":   self.n_features,
            "seq_len":      self.seq_len,
            "horizon":      self.horizon,
            "patch_size":   self.patch_size,
            "stride":       self.stride,
            "d_model":      self.d_model,
            "n_heads":      self.n_heads,
            "n_layers":     self.n_layers,
            "d_ff":         self.d_ff,
            "dropout":      0.0,
            "revin_affine": self.revin_affine,
        }

    def count_parameters(self) -> dict:
        def n(m): return sum(p.numel() for p in m.parameters())
        total = n(self)
        return {
            "revin":                 n(self.revin),
            "patch_embedding":       n(self.patch_embed),
            "temporal_embedding":    n(self.time_embed),
            "positional_encoding":   self.pos_enc.numel(),
            "transformer_encoder":   n(self.encoder) + n(self.encoder_norm),
            "prediction_head":       n(self.head),
            "total":                 total,
            "size_mb":               round(total * 4 / 1024 / 1024, 3),
        }

    def __repr__(self):
        total = sum(p.numel() for p in self.parameters())
        return (
            f"StockForecastNet V5("
            f"features={self.n_features}, seq={self.seq_len}, "
            f"horizon={self.horizon}, patch={self.patch_size}/{self.stride}, "
            f"d_model={self.d_model}, layers={self.n_layers}, "
            f"patches={self.n_patches}, "
            f"params={total:,}, {total*4/1024/1024:.3f}MB)"
        )


# ══════════════════════════════════════════════════════════════════════
# Backwards compatibility aliases
# ══════════════════════════════════════════════════════════════════════
StockPredictor    = StockForecastNet
StockPredictorTFT = StockForecastNet
StockTransformerV2 = StockForecastNet

'''
(PROJECT_ROOT / "model_v2.py").write_text(_model_source)
print(f"✓ model_v2.py written ({len(_model_source):,} chars)")

# Quick import test
import importlib.util
spec = importlib.util.spec_from_file_location("model_v2", PROJECT_ROOT / "model_v2.py")
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
m = mod.StockForecastNet(n_features=56, seq_len=90, horizon=3,
                          patch_size=16, stride=8, d_model=128)
c = m.count_parameters()
print(f"  Test model: {m}")
print(f"  Params: {c['total']:,} ({c['size_mb']} MB)")
del m, mod


In [ ]:
# ─── 3C: Write features_v2.py ─────────────────────────────────────────────────
_features_source = '''\
"""
features_v2.py — Feature engineering for stock OHLCV data
===========================================================
Integrates all 19 strategy modules into a single 60-feature stationary dataset.

DESIGN PRINCIPLES
─────────────────
1. ALL features are stationary.
   Raw prices (open/high/low/close) are NON-STATIONARY — their mean changes
   over time (RELIANCE went from ₹300 in 2010 to ₹2500 today).
   StandardScaler/RobustScaler fitted on training data will produce wildly
   wrong normalisations for future data.
   FIX: every feature is a RATIO, RETURN, or NORMALISED VALUE that has
   the same statistical distribution in 2010 and 2025.

2. No lookahead leakage.
   All rolling windows use pandas default (closed=\'right\') which includes
   only rows up to and including the current row.
   All .shift(N) calls use positive N (shift backward = look at past).
   No shift(-N) anywhere.

3. FEATURE_COLS is the single source of truth.
   dataset_v2.py, train_v2.py, api_v2.py, infer.py all import from here.
   Change features here → changes everywhere automatically.

4. NaN handling.
   Rolling windows create NaN in the first N rows. These are dropped at
   the end of add_features_v2(). The scaler is never fit on NaN rows.

LOOKAHEAD AUDIT (all clean):
   pct_change(N)     → only uses past N values
   rolling(N).mean() → closed=\'right\', only past
   ewm(span=N)       → only past (adjust=False)
   shift(1)          → looks at yesterday (past)
   cumsum()          → only accumulates past values
"""

import numpy as np
import pandas as pd
import os
import sys

# Path fix so this file works when imported from any directory
_ROOT = os.path.dirname(os.path.abspath(__file__))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from strategies.stochastic_strategy import StochasticStrategy
from strategies.cci_strategy import CCIStrategy
from strategies.williams_r_strategy import WilliamsRStrategy
from strategies.obv_strategy import OBVStrategy
from strategies.donchian_strategy import DonchianStrategy
from strategies.supertrend_strategy import SuperTrendStrategy
from strategies.keltner_strategy import KeltnerStrategy
from strategies.heikin_ashi_strategy import HeikinAshiStrategy
from strategies.pivot_strategy import PivotStrategy
from strategies.ichimoku_strategy import IchimokuStrategy


# ─── FEATURE_COLS — canonical list, 60 stationary features ────────────────────
# The Variable Selection Network in model_v2.py will learn which of these
# are actually predictive for a given stock. Having more features is fine —
# the VSN suppresses noisy ones automatically.

FEATURE_COLS = [

    # ── RETURNS (6) ───────────────────────────────────────────────────────────
    # All are pct_change = stationary regardless of price level
    "ret_1d",        # today\'s return: (close - prev_close) / prev_close
    "ret_3d",        # 3-day cumulative return
    "ret_5d",        # 1-week return
    "ret_10d",       # 2-week return
    "ret_20d",       # 1-month return
    "log_ret_1d",    # log(close/prev_close) — symmetric, better for modelling

    # ── VOLATILITY (3) ────────────────────────────────────────────────────────
    "vol_5d",        # std(daily returns, 5 days) — short-term choppiness
    "vol_20d",       # std(daily returns, 20 days) — medium-term volatility
    "vol_ratio",     # vol_5d / vol_20d — is volatility expanding or contracting?

    # ── VOLUME RATIOS (3) ─────────────────────────────────────────────────────
    "volume_ratio_5d",   # today\'s volume / 5-day avg — unusual activity?
    "volume_ratio_20d",  # today\'s volume / 20-day avg
    "volume_trend",      # 5-day vol avg / 20-day vol avg — is participation growing?

    # ── MA RATIOS (5) ─────────────────────────────────────────────────────────
    # close/MA - 1 is stationary (mean-reverts around 0)
    "price_to_ma10",  # short-term: +0.02 = price 2% above 10-day MA
    "price_to_ma20",  # medium-term
    "price_to_ma50",  # long-term
    "ma10_to_ma20",   # MA10/MA20 - 1: crossover signal (pos = bullish)
    "ma20_to_ma50",   # MA20/MA50 - 1

    # ── MACD (2) ──────────────────────────────────────────────────────────────
    "macd_norm",       # (EMA12-EMA26) / close — momentum speed, normalised
    "macd_hist_norm",  # (MACD - Signal) / close — is momentum accelerating?

    # ── RSI (3) ───────────────────────────────────────────────────────────────
    "rsi_14",    # standard 14-period RSI (0-100)
    "rsi_7",     # faster 7-period RSI — catches reversals sooner
    "rsi_diff",  # rsi_7 - rsi_14: positive = short-term stronger than medium

    # ── BOLLINGER BANDS (2) ───────────────────────────────────────────────────
    "bb_position",  # (close-lower)/(upper-lower): 0=at lower, 1=at upper band
    "bb_width",     # (upper-lower)/MA20: low = squeeze, breakout likely

    # ── ATR (2) ───────────────────────────────────────────────────────────────
    "atr_pct",    # ATR / close * 100: typical daily move as % of price
    "atr_ratio",  # current ATR / 20-day avg ATR: volatility expanding?

    # ── PRICE STRUCTURE / CANDLE SHAPE (5) ───────────────────────────────────
    "close_to_high",  # (High-Close)/(High-Low): 0 = closed at day\'s high
    "close_to_low",   # (Close-Low)/(High-Low): 0 = closed at day\'s low
    "body_ratio",     # |Close-Open|/(High-Low): 1 = marubozu (full directional)
    "upper_wick",     # wick above body / range: bearish rejection
    "lower_wick",     # wick below body / range: bullish rejection

    # ── BREAKOUT / SUPPORT (3) ────────────────────────────────────────────────
    "pct_from_20d_high",  # (close - 20d_high) / 20d_high: negative = below resistance
    "pct_from_20d_low",   # (close - 20d_low) / 20d_low: positive = above support
    "breakout_flag",      # 1 if close > yesterday\'s 20-day high

    # ── STOCHASTIC (2) ────────────────────────────────────────────────────────
    "stoch_norm",    # %K / 100 (0-1): position in 14-day price range
    "stoch_cross",   # +1 K crossed above D, -1 crossed below (momentum flip)

    # ── CCI (1) ───────────────────────────────────────────────────────────────
    "cci_norm",      # CCI / 200, clipped [-1,+1]: deviation from statistical avg

    # ── WILLIAMS %R (1) ───────────────────────────────────────────────────────
    "williams_r_norm",  # normalised %R (0=oversold, 1=overbought)

    # ── ON-BALANCE VOLUME (2) ─────────────────────────────────────────────────
    "obv_change",    # +1/-1: was today a volume accumulation or distribution day?
    "obv_to_ma20",   # OBV / OBV_MA20 - 1: divergence from trend (stationary)

    # ── DONCHIAN CHANNELS (2) ─────────────────────────────────────────────────
    "don_position",      # where close sits in 20-day channel (0=bottom, 1=top)
    "don_breakout_up",   # 1 if price broke above yesterday\'s 20-day high

    # ── SUPERTREND (2) ────────────────────────────────────────────────────────
    "supertrend_dir",   # +1 = uptrend, -1 = downtrend (very popular in India)
    "supertrend_dist",  # (close - supertrend_line) / close: how far above/below

    # ── HEIKIN-ASHI (2) ───────────────────────────────────────────────────────
    "ha_trend",         # +1 bullish HA candle, -1 bearish
    "ha_body_norm",     # HA body / close: trend strength from smoothed candles

    # ── PIVOT POINTS (3) ──────────────────────────────────────────────────────
    "dist_to_pp",   # (close - pivot_point) / close: above/below fair value
    "dist_to_r1",   # distance to nearest resistance
    "dist_to_s1",   # distance to nearest support

    # ── ICHIMOKU (2) ──────────────────────────────────────────────────────────
    "ichi_above_cloud",  # +1 above cloud (bullish), -1 below (bearish), 0 inside
    "ichi_tk_cross",     # +1 Tenkan crossed above Kijun (buy signal)

    # ── TREND STRENGTH (2) ────────────────────────────────────────────────────
    "adx_proxy",          # simplified ADX: 0-100, high = strong trend
    "trend_consistency",  # % of last 10 days matching 20-day trend direction

    # ── CALENDAR (3) ──────────────────────────────────────────────────────────
    "day_of_week",   # 0=Mon, 1=Fri (normalised): Mon gaps, Fri profit-taking
    "month_norm",    # 0=Jan, 1=Dec: seasonal patterns, Jan effect, Dec selling
    "is_month_end",  # 1 if last 3 days of month: options expiry, rebalancing
]


# ─── Main feature engineering function ────────────────────────────────────────

def add_features_v2(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute all 60 features from a raw OHLCV DataFrame.

    Args:
        df: DataFrame with columns [open, high, low, close, volume].
            \'datetime\' column or index is used for calendar features if present.

    Returns:
        DataFrame with exactly FEATURE_COLS columns, NaN warmup rows dropped.
        Column order matches FEATURE_COLS exactly.
        All values are stationary — safe to use across any date range.
    """
    df = df.copy()
    _validate_ohlcv(df)

    # Normalise datetime
    if "datetime" not in df.columns and df.index.name == "datetime":
        df = df.reset_index()
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])

    close  = df["close"]
    high   = df["high"]
    low    = df["low"]
    open_  = df["open"]
    volume = df["volume"]

    # ── Returns ───────────────────────────────────────────────────────────────
    df["ret_1d"]     = close.pct_change(1)
    df["ret_3d"]     = close.pct_change(3)
    df["ret_5d"]     = close.pct_change(5)
    df["ret_10d"]    = close.pct_change(10)
    df["ret_20d"]    = close.pct_change(20)
    df["log_ret_1d"] = np.log(close / close.shift(1))

    # ── Volatility ────────────────────────────────────────────────────────────
    ret      = close.pct_change()
    df["vol_5d"]    = ret.rolling(5).std()
    df["vol_20d"]   = ret.rolling(20).std()
    df["vol_ratio"] = df["vol_5d"] / df["vol_20d"].replace(0, np.nan)

    # ── Volume ratios ─────────────────────────────────────────────────────────
    vm5  = volume.rolling(5).mean().replace(0, np.nan)
    vm20 = volume.rolling(20).mean().replace(0, np.nan)
    df["volume_ratio_5d"]  = volume / vm5
    df["volume_ratio_20d"] = volume / vm20
    df["volume_trend"]     = vm5 / vm20

    # ── MA ratios ─────────────────────────────────────────────────────────────
    ma10 = close.rolling(10).mean()
    ma20 = close.rolling(20).mean()
    ma50 = close.rolling(50).mean()
    df["price_to_ma10"] = close / ma10 - 1
    df["price_to_ma20"] = close / ma20 - 1
    df["price_to_ma50"] = close / ma50.replace(0, np.nan) - 1
    df["ma10_to_ma20"]  = ma10 / ma20 - 1
    df["ma20_to_ma50"]  = ma20 / ma50.replace(0, np.nan) - 1

    # ── MACD ──────────────────────────────────────────────────────────────────
    ema12  = close.ewm(span=12, adjust=False).mean()
    ema26  = close.ewm(span=26, adjust=False).mean()
    macd   = ema12 - ema26
    sig    = macd.ewm(span=9, adjust=False).mean()
    df["macd_norm"]      = macd / close.replace(0, np.nan)
    df["macd_hist_norm"] = (macd - sig) / close.replace(0, np.nan)

    # ── RSI ───────────────────────────────────────────────────────────────────
    df["rsi_14"]   = _rsi(close, 14)
    df["rsi_7"]    = _rsi(close, 7)
    df["rsi_diff"] = df["rsi_7"] - df["rsi_14"]

    # ── Bollinger Bands ───────────────────────────────────────────────────────
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_up  = bb_mid + 2 * bb_std
    bb_dn  = bb_mid - 2 * bb_std
    bb_rng = (bb_up - bb_dn).replace(0, np.nan)
    df["bb_upper"]    = bb_up    # needed by KeltnerStrategy for squeeze
    df["bb_lower"]    = bb_dn
    df["bb_position"] = (close - bb_dn) / bb_rng
    df["bb_width"]    = bb_rng / bb_mid.replace(0, np.nan)

    # ── ATR ───────────────────────────────────────────────────────────────────
    atr = _atr(df, 14)
    df["atr_pct"]   = atr / close.replace(0, np.nan) * 100
    df["atr_ratio"] = atr / atr.rolling(20).mean().replace(0, np.nan)

    # ── Price structure (candle shape) ────────────────────────────────────────
    hl = (high - low).replace(0, np.nan)
    df["close_to_high"] = (high - close) / hl
    df["close_to_low"]  = (close - low)  / hl
    df["body_ratio"]    = (close - open_).abs() / hl
    df["upper_wick"]    = (high - df[["open", "close"]].max(axis=1)) / hl
    df["lower_wick"]    = (df[["open", "close"]].min(axis=1) - low)  / hl

    # ── Breakout ──────────────────────────────────────────────────────────────
    h20 = high.rolling(20).max()
    l20 = low.rolling(20).min()
    df["pct_from_20d_high"] = (close - h20) / h20.replace(0, np.nan)
    df["pct_from_20d_low"]  = (close - l20) / l20.replace(0, np.nan)
    df["breakout_flag"]     = (close > h20.shift(1)).astype(float)

    # ── ADX proxy ─────────────────────────────────────────────────────────────
    up_mv   = high.diff()
    dn_mv   = -low.diff()
    plus_dm = up_mv.where((up_mv > dn_mv) & (up_mv > 0), 0.0)
    minus_dm = dn_mv.where((dn_mv > up_mv) & (dn_mv > 0), 0.0)
    tr_sm   = _atr(df, 1).rolling(14).mean().replace(0, np.nan)
    plus_di  = 100 * plus_dm.rolling(14).mean()  / tr_sm
    minus_di = 100 * minus_dm.rolling(14).mean() / tr_sm
    df["adx_proxy"] = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)

    # ── Trend consistency ─────────────────────────────────────────────────────
    df["trend_consistency"] = (
        (np.sign(df["ret_1d"]) == np.sign(df["ret_20d"]))
        .astype(float).rolling(10).mean()
    )

    # ── Calendar ──────────────────────────────────────────────────────────────
    if "datetime" in df.columns:
        dt = df["datetime"]
        df["day_of_week"]  = dt.dt.dayofweek / 4.0
        df["month_norm"]   = (dt.dt.month - 1) / 11.0
        df["is_month_end"] = (
            dt.dt.is_month_end | (dt.dt.day >= dt.dt.days_in_month - 2)
        ).astype(float)
    else:
        df["day_of_week"] = 0.0
        df["month_norm"]  = 0.0
        df["is_month_end"] = 0.0

    # ── Apply strategy modules ─────────────────────────────────────────────────
    # Each strategy adds its own columns; we then pick what we need in FEATURE_COLS

    df = StochasticStrategy(k_period=14, d_period=3).apply(df)
    df = CCIStrategy(period=20).apply(df)
    df = WilliamsRStrategy(period=14).apply(df)
    df = OBVStrategy(ma_period=20).apply(df)
    df = DonchianStrategy(period=20).apply(df)
    df = SuperTrendStrategy(period=10, multiplier=3.0).apply(df)
    df = KeltnerStrategy(ema_period=20, atr_period=10, multiplier=2.0).apply(df)
    df = HeikinAshiStrategy().apply(df)
    df = PivotStrategy(threshold_pct=0.005).apply(df)
    df = IchimokuStrategy(tenkan=9, kijun=26, senkou_b=52).apply(df)

    # ── Drop NaN warmup rows ──────────────────────────────────────────────────
    # Ichimoku(52) + shift(26) needs 78 rows warmup — this is the longest
    before = len(df)
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    print(
        f"[features] {before:,} rows → {len(df):,} clean rows "
        f"(dropped {before - len(df)} NaN warmup rows, "
        f"need ~100 rows minimum per stock)"
    )

    return df[FEATURE_COLS]


# ─── Helpers ──────────────────────────────────────────────────────────────────

def _validate_ohlcv(df: pd.DataFrame):
    required = {"open", "high", "low", "close", "volume"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(
            f"DataFrame missing columns: {missing}\n"
            f"Available columns: {sorted(df.columns.tolist())}"
        )


def _rsi(close: pd.Series, period: int) -> pd.Series:
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def _atr(df: pd.DataFrame, period: int) -> pd.Series:
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift(1)).abs()
    lc = (df["low"]  - df["close"].shift(1)).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr if period == 1 else tr.rolling(period).mean()

'''
(PROJECT_ROOT / "features_v2.py").write_text(_features_source)
print(f"✓ features_v2.py written ({len(_features_source):,} chars)")

# Test import
spec = importlib.util.spec_from_file_location("features_v2", PROJECT_ROOT / "features_v2.py")
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
print(f"  FEATURE_COLS: {len(mod.FEATURE_COLS)} features")
del mod


In [ ]:
# ─── 3D: Write dataset_v2.py ──────────────────────────────────────────────────
_dataset_source = '''\
"""
dataset_v2.py — StockDataset V5
==================================

KEY CHANGES FROM V4
────────────────────
1. Multi-step horizon labels
   V4: y_ret = single scalar (cumulative N-day return)
   V5: y_seq = (horizon,) tensor of cumulative returns from t to t+h
       y_seq[0] = (price[t+1] - price[t]) / price[t]   ← 1-day ahead
       y_seq[1] = (price[t+2] - price[t]) / price[t]   ← 2-day ahead
       y_seq[2] = (price[t+3] - price[t]) / price[t]   ← 3-day ahead (primary signal)

   This gives the model explicit supervision at every horizon step, not just
   the final one. Leads to faster convergence and better calibration.

2. Cyclic temporal features
   V4: calendar features were part of FEATURE_COLS (normalised month_norm etc.)
   V5: calendar features are extracted separately as a second tensor.
       Shape: (window, 6) containing [month_sin, month_cos, dow_sin, dow_cos, dom_sin, dom_cos]
       These are passed as the second input to StockForecastNet.

   Why separate? The temporal embedding module in V5 needs raw cyclic
   values, not pre-normalised scalar values. sin/cos encoding preserves
   the circular structure (December → January is a small step, not a jump
   from 11 to 0 after normalisation).

   If datetime information is not available, time_features is all-zeros.

3. RobustScaler is still used as global pre-processing
   ReVIN in the model handles per-instance non-stationarity.
   The global RobustScaler still serves a purpose: it removes extreme
   outliers (crash days ±10% returns become manageable values) before
   the data enters ReVIN\'s per-instance normalisation.
   Both scalers work together: global RobustScaler first, then per-window
   ReVIN inside the model.

4. __getitem__ returns THREE values (not two):
   x:             (window, n_features)     — scaled features
   time_features: (window, 6)              — cyclic time encodings
   y_seq:         (horizon,)               — multi-step return labels
"""

import glob
import math
import os
import sys

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import RobustScaler
from torch.utils.data import ConcatDataset, Dataset

_ROOT = os.path.dirname(os.path.abspath(__file__))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from features_v2 import FEATURE_COLS, add_features_v2


# ─── Cyclic time encoding helper ──────────────────────────────────────────────

def _cyclic_encode(values: np.ndarray, period: float) -> np.ndarray:
    """
    Encode a periodic variable using sin/cos.

    Example: months 1–12 encoded as sin(2π×month/12), cos(2π×month/12)
    This makes the encoding continuous across boundaries (Dec → Jan).

    Args:
        values: raw integer values (e.g. month 1–12, weekday 0–4)
        period: the full cycle length (12 for months, 5 for trading days)

    Returns:
        (n, 2) array of [sin, cos] encodings, range [-1, +1]
    """
    angle = 2.0 * math.pi * values / period
    return np.stack([np.sin(angle), np.cos(angle)], axis=-1)


def extract_time_features(df: pd.DataFrame, window_start: int, window_len: int) -> np.ndarray:
    """
    Extract cyclic temporal features for rows [window_start : window_start+window_len].

    Returns:
        time_feats: (window_len, 6) array
            Columns: [month_sin, month_cos, dow_sin, dow_cos, dom_sin, dom_cos]

    If \'datetime\' is not in df.columns, returns zeros (model still runs,
    just without temporal context).
    """
    if "datetime" not in df.columns:
        return np.zeros((window_len, 6), dtype=np.float32)

    dt   = pd.to_datetime(df["datetime"].iloc[window_start : window_start + window_len])
    months   = dt.dt.month.values.astype(float)     # 1–12
    weekdays = dt.dt.dayofweek.values.astype(float) # 0 (Mon) – 4 (Fri, ignoring weekends)
    dom      = dt.dt.day.values.astype(float)       # 1–31

    month_enc = _cyclic_encode(months,   12.0)   # (T, 2)
    dow_enc   = _cyclic_encode(weekdays,  5.0)   # (T, 2)
    dom_enc   = _cyclic_encode(dom,      31.0)   # (T, 2)

    return np.concatenate([month_enc, dow_enc, dom_enc], axis=-1).astype(np.float32)
    # → (T, 6)


# ─── Dataset ──────────────────────────────────────────────────────────────────

class StockDatasetV2(Dataset):
    """
    Sliding-window dataset for StockForecastNet V5.

    Each sample is a (window, horizon) pair:
        x             (window, n_features): scaled feature sequences
        time_features (window, 6):          cyclic temporal encodings
        y_seq         (horizon,):           multi-step cumulative return labels

    y_seq[h] = (close[i+window+h+1] - close[i+window]) / close[i+window]

    i.e. the cumulative return from the LAST day of the window to each
    future day. y_seq[0] is the 1-day ahead, y_seq[-1] is the N-day ahead.

    The primary training signal is y_seq[-1] (the furthest horizon),
    but the model is also supervised at intermediate steps.
    """

    def __init__(
        self,
        df,
        window:          int   = 90,     # sequence length (60–90 days recommended)
        horizon:         int   = 3,      # number of future steps (3–5 days)
        noise_threshold: float = 0.001,  # skip samples where |y_seq[-1]| < threshold
        scaler                 = None,   # fitted RobustScaler or None to fit new
        symbol:          str   = "",     # for summary display
    ):
        """
        Args:
            df:              DataFrame with FEATURE_COLS (from add_features_v2)
                             and optionally a \'datetime\' column.
            window:          How many past days to include in each sample.
            horizon:         How many future days to predict (multi-step labels).
            noise_threshold: Skip samples where the primary label |y[-1]| is too
                             small to be meaningful. Reduces noise in training.
            scaler:          Pre-fitted RobustScaler. If None, fit on this df.
            symbol:          Optional name for summary prints.
        """
        missing = set(FEATURE_COLS) - set(df.columns)
        if missing:
            raise ValueError(
                f"DataFrame missing feature columns: {missing}\n"
                "Call add_features_v2(df) before creating StockDatasetV2."
            )

        # ── Store close prices for backtest mark-to-market ─────────────────
        # Before narrowing to FEATURE_COLS, save the raw close price series.
        # The backtest uses these to compute share quantities and entry/exit prices.
        if "close" in df.columns:
            self._close_prices = df["close"].values.copy()
        else:
            self._close_prices = None

        # ── Also store datetime for temporal feature extraction ─────────────
        self._has_datetime = "datetime" in df.columns
        self._df_ref = df   # keep reference for extract_time_features

        # ── Scale features ─────────────────────────────────────────────────
        df_feat = df[FEATURE_COLS].copy().reset_index(drop=True)

        if scaler is None:
            scaler = RobustScaler()
            data   = scaler.fit_transform(df_feat.values)
        else:
            data   = scaler.transform(df_feat.values)

        self.scaler  = scaler
        self.symbol  = symbol
        self.window  = window
        self.horizon = horizon

        # ret_1d values for multi-step label computation (unscaled)
        ret1d = df["ret_1d"].values   # already a return, no need for raw close

        # Close prices for computing cumulative returns from window end
        # Use ret_1d to reconstruct relative price: P[t+h]/P[t] = prod(1+ret[k])
        # Actually we need the close prices to compute y[h] = (P[t+h] - P[t]) / P[t]
        # P is not in FEATURE_COLS, so we use ret_1d to chain returns
        # P[t+h]/P[t] = prod_{k=1}^{h} (1 + ret_1d[t+k])
        # y[h] = P[t+h]/P[t] - 1

        n = len(data)

        X_list    = []
        time_list = []
        y_list    = []

        for i in range(n - window - horizon):
            # Input window: features [i .. i+window-1]
            x = data[i : i + window]     # (window, n_features)

            # Multi-step labels: cumulative returns from position i+window
            labels = np.zeros(horizon, dtype=np.float32)
            valid  = True
            cum    = 1.0

            for h in range(1, horizon + 1):
                r = ret1d[i + window + h - 1]
                if np.isnan(r) or np.isinf(r):
                    valid = False
                    break
                cum        *= (1.0 + r)
                labels[h - 1] = float(cum - 1.0)
                # labels[0] = 1-day cumulative return from window end
                # labels[1] = 2-day cumulative return from window end
                # labels[h-1] = h-day cumulative return from window end

            if not valid:
                continue

            # Skip if primary label (furthest horizon) is near zero
            if abs(labels[-1]) < noise_threshold:
                continue

            # Temporal features for this window
            tf = extract_time_features(df, window_start=i, window_len=window)

            X_list.append(x.astype(np.float32))
            time_list.append(tf)
            y_list.append(labels)

        if len(X_list) == 0:
            raise ValueError(
                f"No valid samples (symbol=\'{symbol}\', window={window}, "
                f"horizon={horizon}, noise_threshold={noise_threshold}). "
                f"Data rows: {n}. Try a longer date range or lower noise_threshold."
            )

        self.X             = torch.tensor(np.array(X_list),    dtype=torch.float32)
        self.time_features = torch.tensor(np.array(time_list), dtype=torch.float32)
        self.y_seq         = torch.tensor(np.array(y_list),    dtype=torch.float32)
        self.n_features    = self.X.shape[2]

        # For internal use (summary, backtest)
        self._primary_labels = self.y_seq[:, -1]   # furthest horizon step

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        """
        Returns: (x, time_features, y_seq)
            x:             (window, n_features)  — scaled feature window
            time_features: (window, 6)            — cyclic time encodings
            y_seq:         (horizon,)             — multi-step return labels
        """
        return self.X[idx], self.time_features[idx], self.y_seq[idx]

    def summary(self):
        n     = len(self.X)
        up    = (self._primary_labels > 0).sum().item()
        down  = n - up
        sym   = f"[{self.symbol}] " if self.symbol else ""
        print(
            f"  {sym}n={n:,}  "
            f"UP={up} ({up/n:.1%})  DOWN={down} ({down/n:.1%})  "
            f"horizon={self.horizon}d  shape={tuple(self.X.shape)}  "
            f"time_shape={tuple(self.time_features.shape)}",
            flush=True,
        )

    def save_scaler(self, path: str):
        joblib.dump(self.scaler, path)
        print(f"  Scaler saved → {path}", flush=True)

    @staticmethod
    def load_scaler(path: str) -> RobustScaler:
        return joblib.load(path)


# ─── Multi-stock dataset builder ──────────────────────────────────────────────

def build_multi_stock_dataset(
    symbols:         list,
    data_dir:        str   = "data",
    window:          int   = 90,
    horizon:         int   = 3,
    noise_threshold: float = 0.001,
    val_split:       float = 0.2,
    gap:             int   = 10,
) -> tuple:
    """
    Build combined train + val datasets from multiple NSE stock symbols.

    Uses a single shared RobustScaler fitted on ALL training data combined.
    This ensures consistent scaling across stocks at the global level,
    while ReVIN in the model handles per-window non-stationarity.

    Args:
        symbols:         NSE stock tickers (["RELIANCE", "TCS", ...])
        data_dir:        Root data directory (contains data/{SYMBOL}/ folders)
        window:          Input sequence length (60–90 days)
        horizon:         Prediction horizon steps (3–5)
        noise_threshold: Min |primary label| to include sample
        val_split:       Fraction of each stock for validation
        gap:             Days to skip between train and val splits

    Returns:
        (train_dataset, val_dataset, fitted_scaler)
    """
    print(f"\nBuilding multi-stock dataset: {symbols}", flush=True)

    train_dfs = []
    val_dfs   = []

    for symbol in symbols:
        folder = os.path.join(data_dir, symbol.upper())
        files  = sorted(glob.glob(os.path.join(folder, "*.parquet")))
        if not files:
            print(f"  [SKIP] {symbol}: no .parquet in {folder}", flush=True)
            continue

        df_raw = pd.read_parquet(files[-1])
        print(f"  {symbol}: {len(df_raw):,} raw rows", flush=True)

        df = add_features_v2(df_raw)

        n       = len(df)
        n_val   = int(n * val_split)
        n_train = n - n_val - gap

        if n_train < window + horizon + 50:
            print(f"  [SKIP] {symbol}: only {n_train} training rows", flush=True)
            continue

        train_dfs.append((symbol, df.iloc[:n_train].copy()))
        val_dfs.append((symbol, df.iloc[n_train + gap:].copy()))

    if not train_dfs:
        raise ValueError(
            "No usable stock data. "
            "Run: python data_fetch_upstox.py --symbol RELIANCE --start 2010-01-01"
        )

    # Fit ONE scaler on ALL training data
    combined = np.vstack([df[FEATURE_COLS].values for _, df in train_dfs])
    scaler   = RobustScaler()
    scaler.fit(combined)
    print(f"\n  Shared scaler fitted on {len(combined):,} rows from {len(train_dfs)} stocks", flush=True)

    train_sets, val_sets = [], []

    for symbol, df in train_dfs:
        try:
            ds = StockDatasetV2(df, window=window, horizon=horizon,
                                noise_threshold=noise_threshold,
                                scaler=scaler, symbol=symbol)
            ds.summary()
            train_sets.append(ds)
        except ValueError as e:
            print(f"  [SKIP train] {symbol}: {e}", flush=True)

    for symbol, df in val_dfs:
        try:
            ds = StockDatasetV2(df, window=window, horizon=horizon,
                                noise_threshold=noise_threshold,
                                scaler=scaler, symbol=symbol)
            val_sets.append(ds)
        except ValueError as e:
            print(f"  [SKIP val] {symbol}: {e}", flush=True)

    if not train_sets:
        raise ValueError("All stocks failed. Check data and parameters.")

    train_combined = ConcatDataset(train_sets)
    val_combined   = ConcatDataset(val_sets)

    n_tr = sum(len(d) for d in train_sets)
    n_va = sum(len(d) for d in val_sets)
    print(
        f"\n  TOTAL: {n_tr:,} train + {n_va:,} val samples "
        f"from {len(train_sets)} stocks\n",
        flush=True,
    )

    return train_combined, val_combined, scaler

'''
(PROJECT_ROOT / "dataset_v2.py").write_text(_dataset_source)
print(f"✓ dataset_v2.py written ({len(_dataset_source):,} chars)")


In [ ]:
# ─── 3E: Write utils/trading_v2.py and utils/__init__.py ──────────────────────
_trading_source = '''\
"""
utils/trading_v2.py — Signal generation for StockPredictor V4
==============================================================

WHY THE OLD BACKTEST TRADED NOTHING
─────────────────────────────────────
The old `generate_signal_v2` expected:
    direction  = int from argmax(softmax(logits))  — 0 or 1
    confidence = float from softmax.max()           — 0.0 to 1.0

With the V4 single-head model, there are NO logits. The model outputs
a signed return scalar. Confidence is derived from the magnitude of that
scalar via sigmoid. With typical model outputs of ±0.005 to ±0.02:
    sigmoid(0.01 × 100) = sigmoid(1.0) = 0.73

So confidence CAN reach above the old thresholds — BUT the old
`generate_signal_v2` was still called with `confidence` from a
misinterpreted float and `direction` from `argmax` of a 1-dim tensor,
producing unpredictable results.

V4 SIGNAL LOGIC
────────────────
The V4 model outputs a single signed return prediction:
    pred > 0  → model predicts price will go UP over the horizon
    pred < 0  → model predicts price will go DOWN
    |pred|    → magnitude of the expected move

Confidence is derived from magnitude:
    conf = sigmoid(|pred| × CONF_SCALE)

    |pred| = 0.005 (0.5%)   → conf = sigmoid(0.5)  = 0.62
    |pred| = 0.010 (1.0%)   → conf = sigmoid(1.0)  = 0.73
    |pred| = 0.020 (2.0%)   → conf = sigmoid(2.0)  = 0.88
    |pred| = 0.030 (3.0%)   → conf = sigmoid(3.0)  = 0.95

Threshold tuning:
    CONF_SCALE is the key parameter. Higher = more selective (fewer trades).
    CONFIDENCE_FLOOR is the minimum confidence to generate ANY signal.

    For a 3-day horizon on a model that just started training:
    - Expected |pred| at random initialisation: ~0.001-0.005
    - conf from random model: ~0.50-0.62  (below CONFIDENCE_FLOOR=0.52)
    - conf from trained model: 0.60-0.88  (above floor, generates signals)

    This means: a random model produces HOLD (good — no spurious trades).
    A trained model produces BUY/SELL signals.

THRESHOLDS
───────────
CONFIDENCE_FLOOR = 0.52  (lowered from 0.55 — allows more trades in backtest)
CONF_SCALE = 100.0       (sigmoid scaling factor for magnitude → confidence)

For 3-day horizon, a 1% predicted move is meaningful.
For 1-day horizon, raise thresholds (more noise in short horizon).
"""

import math
from dataclasses import dataclass
from typing import Literal

# ─── Scaling and threshold constants ─────────────────────────────────────────

# CONF_SCALE: multiplier applied to |pred_return| before sigmoid
# Calibrated so that |pred| = 1% gives confidence ≈ 0.73
# Increase to be more selective (fewer trades); decrease for more trades
CONF_SCALE = 100.0

# Floor: below this confidence → always HOLD
# Lowered from 0.55 to 0.52 — allows more backtest trades
# Raise to 0.60+ for live trading with real capital
CONFIDENCE_FLOOR = 0.52

# Signal strength tiers
STRONG_CONFIDENCE = 0.70   # sigmoid(|pred|×100) ≥ 0.70 → |pred| ≥ 0.85%
MEDIUM_CONFIDENCE = 0.60   # sigmoid(|pred|×100) ≥ 0.60 → |pred| ≥ 0.40%

# Return magnitude thresholds (additional filter on top of confidence)
STRONG_RETURN_PCT = 0.010  # predicted move ≥ 1.0% for STRONG
MEDIUM_RETURN_PCT = 0.004  # predicted move ≥ 0.4% for MEDIUM


@dataclass
class SignalResult:
    signal:     Literal["BUY", "SELL", "HOLD"]
    strength:   Literal["STRONG", "MEDIUM", "WEAK"]
    reason:     str


def pred_to_confidence(pred_return: float, scale: float = CONF_SCALE) -> float:
    """
    Convert a raw signed return prediction to a confidence value in [0, 1].

    Uses sigmoid of the magnitude:
        conf = 1 / (1 + exp(-|pred_return| × scale))

    Args:
        pred_return: signed return from model (e.g. +0.015 = +1.5%)
        scale:       scaling factor (default CONF_SCALE=100)

    Returns:
        confidence in (0.5, 1.0) — always above 0.5 since we take |pred|
    """
    magnitude = abs(pred_return)
    return 1.0 / (1.0 + math.exp(-magnitude * scale))


def generate_signal_v2(
    direction:       int,
    confidence:      float,
    expected_return: float,
) -> tuple[str, str]:
    """
    Convert model outputs into a trading signal.

    For V4 single-head model, call like this:
        pred = model(x).item()                          # signed return
        direction = 1 if pred > 0 else 0               # derived direction
        confidence = pred_to_confidence(pred)           # magnitude-based
        signal, strength = generate_signal_v2(direction, confidence, pred)

    Args:
        direction:       1 = UP prediction, 0 = DOWN prediction
        confidence:      0-1 value (from pred_to_confidence or softmax)
        expected_return: raw predicted return (signed float)

    Returns:
        (signal, strength) where:
            signal   ∈ {BUY, SELL, HOLD}
            strength ∈ {STRONG, MEDIUM, WEAK}
    """
    result = _evaluate(direction, confidence, expected_return)
    return result.signal, result.strength


def _evaluate(direction: int, confidence: float,
              expected_return: float) -> SignalResult:

    abs_ret = abs(expected_return)

    # Gate 1: minimum confidence floor
    if confidence < CONFIDENCE_FLOOR:
        return SignalResult(
            "HOLD", "WEAK",
            f"conf={confidence:.3f} below floor={CONFIDENCE_FLOOR}"
        )

    # Gate 2: minimum return magnitude (filters noise from near-zero predictions)
    if abs_ret < MEDIUM_RETURN_PCT:
        return SignalResult(
            "HOLD", "WEAK",
            f"|ret|={abs_ret:.4f} below minimum={MEDIUM_RETURN_PCT}"
        )

    # BUY signals
    if direction == 1:
        if confidence >= STRONG_CONFIDENCE and abs_ret >= STRONG_RETURN_PCT:
            return SignalResult(
                "BUY", "STRONG",
                f"conf={confidence:.3f} pred={expected_return:+.4f}"
            )
        if confidence >= MEDIUM_CONFIDENCE:
            return SignalResult(
                "BUY", "MEDIUM",
                f"conf={confidence:.3f} pred={expected_return:+.4f}"
            )
        return SignalResult(
            "HOLD", "WEAK",
            f"BUY: conf={confidence:.3f} below MEDIUM_CONFIDENCE={MEDIUM_CONFIDENCE}"
        )

    # SELL signals
    if direction == 0:
        if confidence >= STRONG_CONFIDENCE and abs_ret >= STRONG_RETURN_PCT:
            return SignalResult(
                "SELL", "STRONG",
                f"conf={confidence:.3f} pred={expected_return:+.4f}"
            )
        if confidence >= MEDIUM_CONFIDENCE:
            return SignalResult(
                "SELL", "MEDIUM",
                f"conf={confidence:.3f} pred={expected_return:+.4f}"
            )
        return SignalResult(
            "HOLD", "WEAK",
            f"SELL: conf={confidence:.3f} below MEDIUM_CONFIDENCE={MEDIUM_CONFIDENCE}"
        )

    return SignalResult("HOLD", "WEAK", "Unknown direction")

'''
utils_dir = PROJECT_ROOT / "utils"
(utils_dir / "trading_v2.py").write_text(_trading_source)
(utils_dir / "__init__.py").write_text(
    "from .trading_v2 import generate_signal_v2, pred_to_confidence, CONFIDENCE_FLOOR\n"
    "__all__ = ['generate_signal_v2', 'pred_to_confidence', 'CONFIDENCE_FLOOR']\n"
)
print(f"✓ utils/trading_v2.py written ({len(_trading_source):,} chars)")
print(f"✓ utils/__init__.py written")


In [ ]:
# ─── 3F: Write backtest_v2.py ─────────────────────────────────────────────────
_backtest_source = '''\
"""
backtest_v2.py — Portfolio-Based Backtesting for StockForecastNet V5
====================================================================

TRADING RULES IMPLEMENTED
───────────────────────────
Rule 1 — Current Holdings:
    Portfolio tracks actual SHARE QUANTITIES, not abstract ₹ amounts.
    Every day, all held shares are marked to market at the previous day\'s
    closing price, and unrealised P&L is included in total portfolio value.

Rule 2 — Buy Condition:
    A BUY signal allocates position_size_pct × available_cash.
    If available cash < min_trade_value, the buy is SKIPPED (insufficient funds).
    Shares bought = floor(allocated_cash / entry_price)
    Leftover cash (from rounding) stays in the cash account.

Rule 3 — Sell Condition:
    A SELL signal checks whether we currently hold shares of this stock.
    If shares_held == 0, the signal is IGNORED (cannot short what you don\'t own).
    If shares_held > 0, ALL held shares are sold at the current close price.
    (Short-selling is not modelled — this is a long-only portfolio.)

Rule 4 — P&L Calculation:
    Realised P&L:   profit/loss locked in from completed sell transactions.
    Unrealised P&L: value of currently held shares minus their purchase cost,
                    using the most recent available close price (prev day close).
    Total P&L       = Realised P&L + Unrealised P&L
    Total Value     = Cash + (shares_held × latest_close_price)

Rule 5 — Portfolio Integrity:
    shares_held is only modified by confirmed buy/sell executions.
    avg_cost_per_share is updated on each buy (weighted average cost).
    On sell, realised P&L = (sell_price - avg_cost) × shares_sold - costs.

WHY PREVIOUS APPROACHES WERE WRONG
────────────────────────────────────
Old approach: pnl = capital × actual_return
  This treats capital as a continuous % bet, not as real shares.
  It ignores the mechanics of buying N shares at price P and selling at P\'.
  Share rounding (you can\'t buy 0.7 shares) and dividend treatment matter.
  More importantly, it allowed "SELL signals" even when nothing was held.

New approach: proper share ledger
  cash                   — INR available for new purchases
  shares_held            — integer number of shares currently owned
  avg_cost_per_share     — weighted average purchase cost
  Realised P&L           — accumulated on every sell
  Unrealised P&L         — computed fresh each day from latest price

USAGE
──────
  # Basic (no per-trade log)
  python backtest_v2.py --data data/RELIANCE/RELIANCE_daily_2010-01-01_2026-04-09.parquet

  # With per-trade logging
  python backtest_v2.py --data ... --log_trades

  # Save full trade log to CSV (open in Excel)
  python backtest_v2.py --data ... --log_trades --csv trades.csv

  # Adjust position size (default 20% of cash per buy signal)
  python backtest_v2.py --data ... --position_size 0.10

  # Lower confidence threshold (more signals)
  python backtest_v2.py --data ... --confidence 0.50
"""

import argparse
import csv
import os
import sys
from dataclasses import dataclass, field
from typing import List, Optional

_ROOT = os.path.dirname(os.path.abspath(__file__))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

import joblib
import numpy as np
import torch

from dataset_v2 import StockDatasetV2
from features_v2 import add_features_v2, FEATURE_COLS
from model_v2 import StockForecastNet
StockPredictor = StockForecastNet  # backwards compat
from utils.trading_v2 import (
    CONFIDENCE_FLOOR, generate_signal_v2, pred_to_confidence
)


# ─── Trading cost constants (NSE realistic) ───────────────────────────────────
BROKERAGE_PCT  = 0.001    # 0.10% per leg (Zerodha/Upstox flat fee approximation)
SLIPPAGE_PCT   = 0.0005   # 0.05% slippage per leg (market impact + bid-ask spread)
COST_PER_LEG   = BROKERAGE_PCT + SLIPPAGE_PCT   # 0.15% per leg
MIN_TRADE_VALUE = 1_000.0  # ₹1,000 minimum trade size (avoid micro lots)


# ─── Data structures ──────────────────────────────────────────────────────────

@dataclass
class Position:
    """
    Represents one open stock holding in the portfolio.

    shares:          Number of whole shares currently held.
    avg_cost:        Weighted average purchase price per share (includes entry costs).
    total_cost_basis: Total INR spent to acquire this position (for realised P&L calc).
    open_day_idx:    Dataset index when this position was opened.
    signal:          "BUY" (long) — short selling not modelled.
    pred_return:     What the model predicted at entry.
    confidence:      Model confidence at entry.
    strength:        "STRONG" or "MEDIUM".
    entry_price:     Actual close price at entry.
    """
    shares:           int
    avg_cost:         float
    total_cost_basis: float
    open_day_idx:     int
    signal:           str
    pred_return:      float
    confidence:       float
    strength:         str
    entry_price:      float


@dataclass
class ClosedTrade:
    """
    Complete record of one round-trip trade (buy → sell).
    Used for the final summary report and CSV export.
    """
    trade_num:       int
    open_day_idx:    int
    close_day_idx:   int
    signal:          str         # always "BUY" in long-only mode
    strength:        str
    pred_return:     float       # model prediction at entry
    confidence:      float
    entry_price:     float       # ₹ per share at buy
    exit_price:      float       # ₹ per share at sell
    shares:          int
    gross_pnl:       float       # (exit_price - entry_price) × shares
    costs:           float       # total brokerage + slippage (both legs)
    net_pnl:         float       # gross_pnl - costs
    is_win:          bool
    cash_before:     float       # portfolio cash before the sell executed
    cash_after:      float       # portfolio cash after the sell executed
    portfolio_value_after: float  # total portfolio value after close


@dataclass
class Portfolio:
    """
    Complete portfolio state.

    cash:              INR not invested in any position.
    position:          The currently open position (None if flat).
    closed_trades:     List of all completed round trips.
    realised_pnl:      Sum of all net_pnl from closed trades.
    best_streak:       Longest consecutive winning streak.
    worst_streak:      Longest consecutive losing streak.
    _current_streak:   Running streak counter (+ve=wins, -ve=losses).
    """
    initial_capital:  float
    cash:             float
    position:         Optional[Position] = None
    closed_trades:    List[ClosedTrade]  = field(default_factory=list)
    realised_pnl:     float = 0.0
    best_streak:      int   = 0
    worst_streak:     int   = 0
    _current_streak:  int   = 0

    @property
    def shares_held(self) -> int:
        return self.position.shares if self.position else 0

    def total_value(self, current_price: float) -> float:
        """
        Total portfolio value = cash + market value of any open position.

        Uses current_price (latest available close) to mark the position
        to market. This is the unrealised P&L calculation.
        """
        if self.position:
            return self.cash + self.position.shares * current_price
        return self.cash

    def unrealised_pnl(self, current_price: float) -> float:
        """
        Unrealised P&L on the open position at the current price.
        = (current_price - avg_cost) × shares  [before exit costs]
        """
        if not self.position:
            return 0.0
        return (current_price - self.position.avg_cost) * self.position.shares

    def update_streak(self, is_win: bool):
        if is_win:
            self._current_streak = max(self._current_streak + 1, 1)
            self.best_streak = max(self.best_streak, self._current_streak)
        else:
            self._current_streak = min(self._current_streak - 1, -1)
            self.worst_streak = min(self.worst_streak, self._current_streak)


# ─── Core execution functions ──────────────────────────────────────────────────

def _execute_buy(
    portfolio:         Portfolio,
    day_idx:           int,
    entry_price:       float,
    pred_return:       float,
    confidence:        float,
    strength:          str,
    position_size_pct: float,
    log_trades:        bool,
    day_label:         str,
) -> bool:
    """
    Attempt to open a long position.

    Rule 2 — Buy Condition:
        Only executes if:
          (a) No position currently held (one position at a time)
          (b) Available cash >= MIN_TRADE_VALUE after deducting costs

    Share quantity:
        allocated = cash × position_size_pct
        entry_cost_rate = COST_PER_LEG (brokerage + slippage on entry)
        effective_price = entry_price × (1 + COST_PER_LEG)
        shares = floor(allocated / effective_price)

        The effective_price is higher than the raw close price because
        we pay brokerage and slippage on the way in. This ensures
        the cost_basis accurately reflects what we actually spent.

    Returns True if the buy was executed, False if skipped.
    """
    # Rule 2: check no open position
    if portfolio.position is not None:
        return False

    allocated = portfolio.cash * position_size_pct

    # Rule 2: check sufficient capital
    if allocated < MIN_TRADE_VALUE:
        if log_trades:
            print(
                f"  {day_label}  SKIP BUY — insufficient cash "
                f"(₹{portfolio.cash:,.0f} × {position_size_pct:.0%} = "
                f"₹{allocated:,.0f} < min ₹{MIN_TRADE_VALUE:,.0f})",
                flush=True,
            )
        return False

    if entry_price <= 0:
        return False

    # Effective price per share including entry costs
    effective_price = entry_price * (1.0 + COST_PER_LEG)
    shares = int(allocated / effective_price)   # floor — only whole shares

    if shares <= 0:
        return False

    # Actual cash deducted = shares × effective_price (includes brokerage+slippage)
    cash_deducted = shares * effective_price
    portfolio.cash -= cash_deducted

    portfolio.position = Position(
        shares           = shares,
        avg_cost         = effective_price,     # cost per share inc. entry costs
        total_cost_basis = cash_deducted,
        open_day_idx     = day_idx,
        signal           = "BUY",
        pred_return      = pred_return,
        confidence       = confidence,
        strength         = strength,
        entry_price      = entry_price,
    )

    if log_trades:
        print(
            f"  {day_label}  ▲ BUY   {strength:>6}  "
            f"pred={pred_return:+.2%}  conf={confidence:.3f}  "
            f"price=₹{entry_price:>8,.2f}  "
            f"shares={shares:>5}  "
            f"cost_basis=₹{cash_deducted:>10,.2f}  "
            f"cash_left=₹{portfolio.cash:>10,.2f}",
            flush=True,
        )

    return True


def _execute_sell(
    portfolio:    Portfolio,
    day_idx:      int,
    exit_price:   float,
    trade_num:    int,
    log_trades:   bool,
    day_label:    str,
    forced:       bool = False,   # True when closing at end-of-data
) -> Optional[ClosedTrade]:
    """
    Close the open long position.

    Rule 3 — Sell Condition:
        Only executes if portfolio.position is not None (shares exist).
        If no position is held, signal is ignored and None is returned.

    Rule 4 — Realised P&L:
        gross_pnl = (exit_price - entry_price) × shares
        exit_cost = shares × exit_price × COST_PER_LEG
        net_pnl   = gross_pnl - exit_cost
        (entry cost is already embedded in cost_basis / avg_cost)

    Cash after sell:
        cash += shares × exit_price × (1 - COST_PER_LEG)

    Returns the ClosedTrade record, or None if no position to close.
    """
    # Rule 3: nothing to sell
    if portfolio.position is None:
        if log_trades and not forced:
            print(
                f"  {day_label}  SKIP SELL — no open position (short-selling not allowed)",
                flush=True,
            )
        return None

    pos       = portfolio.position
    shares    = pos.shares

    # Proceeds received after exit costs (slippage + brokerage on the way out)
    exit_cost_rate = COST_PER_LEG
    proceeds   = shares * exit_price * (1.0 - exit_cost_rate)
    exit_cost  = shares * exit_price * exit_cost_rate

    # Entry cost is already in cost_basis — total cost = entry cost + exit cost
    entry_cost = pos.total_cost_basis - (shares * pos.entry_price)  # cost portion only
    total_cost = entry_cost + exit_cost

    # Gross P&L: purely the price difference × shares
    gross_pnl = (exit_price - pos.entry_price) * shares
    # Net P&L: what actually flows into/out of cash
    net_pnl   = proceeds - pos.total_cost_basis   # what we got back minus what we spent

    portfolio.cash        += proceeds
    portfolio.realised_pnl += net_pnl
    portfolio.position    = None   # position is closed

    is_win = net_pnl > 0.0
    portfolio.update_streak(is_win)

    total_value_after = portfolio.total_value(exit_price)

    trade = ClosedTrade(
        trade_num             = trade_num,
        open_day_idx          = pos.open_day_idx,
        close_day_idx         = day_idx,
        signal                = pos.signal,
        strength              = pos.strength,
        pred_return           = pos.pred_return,
        confidence            = pos.confidence,
        entry_price           = pos.entry_price,
        exit_price            = exit_price,
        shares                = shares,
        gross_pnl             = round(gross_pnl, 2),
        costs                 = round(total_cost, 2),
        net_pnl               = round(net_pnl, 2),
        is_win                = is_win,
        cash_before           = round(portfolio.cash - proceeds, 2),
        cash_after            = round(portfolio.cash, 2),
        portfolio_value_after = round(total_value_after, 2),
    )

    if log_trades:
        win_sym = "✓ WIN " if is_win else "✗ LOSS"
        label   = "CLOSE" if not forced else "EXPIRY"
        print(
            f"  {day_label}  ▼ {label:<6}  {pos.strength:>6}  "
            f"entry=₹{pos.entry_price:>8,.2f}  "
            f"exit=₹{exit_price:>8,.2f}  "
            f"shares={shares:>5}  "
            f"net_pnl=₹{net_pnl:>+10,.2f}  "
            f"cash=₹{portfolio.cash:>10,.2f}  "
            f"total=₹{total_value_after:>10,.2f}  "
            f"{win_sym}",
            flush=True,
        )

    return trade


# ─── Main backtest function ───────────────────────────────────────────────────

def backtest_v2(
    model:              StockForecastNet,
    dataset:            StockDatasetV2,
    horizon:            int   = 3,
    min_confidence:     float = CONFIDENCE_FLOOR,
    position_size_pct:  float = 0.20,
    device:             str   = "cpu",
    log_trades:         bool  = True,
    log_interval:       int   = 250,
    csv_path:           Optional[str] = None,
) -> dict:
    """
    Portfolio-based walk-forward backtest.

    DAILY LOOP (for each day i in the dataset):
    ────────────────────────────────────────────
    1. Get today\'s close price from dataset._close_prices.
       This price is used to:
         a) Mark any open position to market (unrealised P&L)
         b) As the exit price if a SELL signal fires today
         c) As the entry price if a BUY signal fires today

    2. Check if an open position should close today.
       The position was opened on day open_day_idx.
       It closes on day open_day_idx + horizon.
       We use the SELL signal logic at close time:
         - Execute sell at today\'s close price
         - Record realised P&L

    3. Run model inference (only if no position is open — no overlap).
       pred_raw → direction → confidence → signal/strength

    4. Execute signal:
       - BUY:  Open new long position (Rule 2 — check cash)
       - SELL: Close open position    (Rule 3 — check holdings)
               In long-only mode, a model SELL signal after horizon days
               means the model predicted DOWN — we use this as the exit trigger.

    5. Snapshot today\'s total portfolio value:
       total_value = cash + shares_held × today_close_price

    Note on horizon-based exit:
    The model was trained to predict the N-day return.
    The "sell after horizon days" rule matches the training objective.
    A model SELL signal on a day we are flat is ignored (long-only).

    Returns: dict with full metrics and trade list.
    """
    dev = torch.device(device)
    model.eval()
    model.to(dev)

    # Validate close prices available
    if dataset._close_prices is None:
        raise ValueError(
            "dataset._close_prices is None — the dataset was built without "
            "a DataFrame containing a \'close\' column. Pass the full featured "
            "df (output of add_features_v2 called on a df that has \'close\')."
        )

    close_prices = dataset._close_prices  # shape: (n_rows_in_clean_df,)
    window       = dataset.X.shape[1]     # e.g. 30
    n_samples    = len(dataset)

    portfolio   = Portfolio(
        initial_capital = 100_000.0,
        cash            = 100_000.0,
    )
    cap_curve   = []      # total portfolio value each day
    trade_count = 0

    _print_header(position_size_pct, min_confidence, horizon)

    for i in range(n_samples):
        # ── Resolve today\'s close price ───────────────────────────────────
        # dataset[i] covers feature rows [i .. i+window-1]
        # The signal fires on row i+window (first day after the feature window)
        # Entry/exit price = close on row (i + window)
        price_idx     = i + window        # index into close_prices array
        horizon_idx   = i + window + horizon   # index of exit day

        # Clamp to valid range
        price_idx_clamped   = min(price_idx,   len(close_prices) - 1)
        horizon_idx_clamped = min(horizon_idx, len(close_prices) - 1)

        today_close  = float(close_prices[price_idx_clamped])
        exit_close   = float(close_prices[horizon_idx_clamped])

        day_label = f"Day {i:>4}/{n_samples}"

        # ── Check if open position should close today ─────────────────────
        # Rule: positions opened on day i_open close on day i_open + horizon
        # We check: current day i == i_open + horizon  →  close the position
        if portfolio.position is not None:
            open_idx = portfolio.position.open_day_idx
            if i >= open_idx + horizon:
                trade_count += 1
                trade = _execute_sell(
                    portfolio   = portfolio,
                    day_idx     = i,
                    exit_price  = today_close,   # close at today\'s price
                    trade_num   = trade_count,
                    log_trades  = log_trades,
                    day_label   = day_label,
                )
                if trade:
                    portfolio.closed_trades.append(trade)

        # ── Snapshot portfolio value (mark to market) ──────────────────────
        # Rule 1 & 4: total value includes unrealised P&L at today\'s close
        total_val = portfolio.total_value(today_close)
        cap_curve.append(total_val)

        # ── Progress log ──────────────────────────────────────────────────
        if log_interval > 0 and i > 0 and i % log_interval == 0:
            ret_pct = (total_val / portfolio.initial_capital - 1) * 100
            unreal  = portfolio.unrealised_pnl(today_close)
            print(
                f"  Day {i:>4}/{n_samples}  "
                f"Total=₹{total_val:>10,.0f} ({ret_pct:+.1f}%)  "
                f"Cash=₹{portfolio.cash:>9,.0f}  "
                f"Shares={portfolio.shares_held:>4}  "
                f"UnrealPnL=₹{unreal:>+9,.0f}  "
                f"Trades={len(portfolio.closed_trades)}",
                flush=True,
            )

        # ── Run model inference ───────────────────────────────────────────
        # Skip if already in a position (no overlapping trades)
        if portfolio.position is not None:
            continue

        x, time_feats, _ = dataset[i]   # V5: unpack (x, time_features, y_ret)
        with torch.no_grad():
            # V5: model takes both feature seq and time features
            # predictions shape: (1, horizon); use last step as primary signal
            preds    = model(x.unsqueeze(0).to(dev), time_feats.unsqueeze(0).to(dev))
            pred_raw = preds[0, -1].item()   # primary signal = furthest horizon step

        direction  = 1 if pred_raw > 0 else 0
        confidence = pred_to_confidence(pred_raw)

        if confidence < min_confidence:
            continue

        signal, strength = generate_signal_v2(direction, confidence, pred_raw)

        # ── Execute signal ────────────────────────────────────────────────
        if signal == "BUY":
            # Rule 2: buy only if sufficient cash
            _execute_buy(
                portfolio          = portfolio,
                day_idx            = i,
                entry_price        = today_close,
                pred_return        = pred_raw,
                confidence         = confidence,
                strength           = strength,
                position_size_pct  = position_size_pct,
                log_trades         = log_trades,
                day_label          = day_label,
            )

        elif signal == "SELL":
            # Rule 3: in long-only mode, SELL means "exit existing position"
            # If we have no position, this signal is ignored
            if portfolio.position is not None:
                trade_count += 1
                trade = _execute_sell(
                    portfolio  = portfolio,
                    day_idx    = i,
                    exit_price = today_close,
                    trade_num  = trade_count,
                    log_trades = log_trades,
                    day_label  = day_label,
                )
                if trade:
                    portfolio.closed_trades.append(trade)
            else:
                if log_trades:
                    print(
                        f"  {day_label}  IGNORE SELL — no position held "
                        f"(long-only: cannot short)",
                        flush=True,
                    )

    # ── Force-close any remaining open position at end of data ────────────
    if portfolio.position is not None:
        last_price = float(close_prices[-1])
        trade_count += 1
        trade = _execute_sell(
            portfolio  = portfolio,
            day_idx    = n_samples - 1,
            exit_price = last_price,
            trade_num  = trade_count,
            log_trades = log_trades,
            day_label  = f"Day {n_samples}/{n_samples} [END]",
            forced     = True,
        )
        if trade:
            portfolio.closed_trades.append(trade)
        # Final snapshot
        cap_curve.append(portfolio.total_value(last_price))

    # ── Build results dict ────────────────────────────────────────────────
    final_close  = float(close_prices[-1])
    final_value  = portfolio.total_value(final_close)
    unrealised   = portfolio.unrealised_pnl(final_close)
    results      = _compute_metrics(portfolio, cap_curve, final_value, final_close)

    _print_results(results, portfolio)

    if csv_path:
        _save_csv(portfolio.closed_trades, csv_path)
        print(f"\n  Trade log saved → {csv_path}", flush=True)

    return results


# ─── Metrics ──────────────────────────────────────────────────────────────────

def _compute_metrics(
    portfolio:   Portfolio,
    cap_curve:   list,
    final_value: float,
    final_price: float,
) -> dict:
    trades    = portfolio.closed_trades
    n_trades  = len(trades)
    wins      = sum(1 for t in trades if t.is_win)
    initial   = portfolio.initial_capital

    strong_t  = [t for t in trades if t.strength == "STRONG"]
    medium_t  = [t for t in trades if t.strength == "MEDIUM"]

    total_costs  = sum(t.costs   for t in trades)
    total_net    = sum(t.net_pnl for t in trades)

    avg_win  = (np.mean([t.net_pnl for t in trades if t.is_win])
                if wins > 0 else 0.0)
    avg_loss = (np.mean([t.net_pnl for t in trades if not t.is_win])
                if (n_trades - wins) > 0 else 0.0)

    pnl_per_trade = [t.net_pnl / max(t.shares * t.entry_price, 1.0) for t in trades]
    sharpe = _sharpe(pnl_per_trade)
    maxdd  = _maxdd(cap_curve)

    unrealised = portfolio.unrealised_pnl(final_price)

    return {
        "initial_capital":    initial,
        "final_value":        round(final_value, 2),
        "cash":               round(portfolio.cash, 2),
        "shares_held":        portfolio.shares_held,
        "unrealised_pnl":     round(unrealised, 2),
        "realised_pnl":       round(portfolio.realised_pnl, 2),
        "total_return_pct":   round((final_value / initial - 1) * 100, 2),
        "total_costs":        round(total_costs, 2),
        "n_days":             len(cap_curve),
        "n_trades":           n_trades,
        "accuracy":           round(wins / n_trades, 4) if n_trades else 0.0,
        "wins":               wins,
        "losses":             n_trades - wins,
        "avg_win_inr":        round(avg_win,  2),
        "avg_loss_inr":       round(avg_loss, 2),
        "profit_factor":      round(abs(avg_win / avg_loss), 3) if avg_loss != 0 else 0.0,
        "sharpe_ratio":       round(sharpe, 3),
        "max_drawdown_pct":   round(maxdd * 100, 2),
        "best_streak":        portfolio.best_streak,
        "worst_streak":       abs(portfolio.worst_streak),
        "n_strong":           len(strong_t),
        "strong_accuracy":    (round(sum(1 for t in strong_t if t.is_win) / len(strong_t), 4)
                               if strong_t else 0.0),
        "n_medium":           len(medium_t),
        "medium_accuracy":    (round(sum(1 for t in medium_t if t.is_win) / len(medium_t), 4)
                               if medium_t else 0.0),
        "capital_curve":      cap_curve,
        "trades":             trades,
    }


def _sharpe(rets: list) -> float:
    if len(rets) < 2: return 0.0
    a = np.array(rets, dtype=np.float64)
    s = a.std()
    return float(np.sqrt(252) * a.mean() / s) if s > 1e-9 else 0.0


def _maxdd(curve: list) -> float:
    if not curve: return 0.0
    arr  = np.array(curve, dtype=np.float64)
    peak = arr[0]; dd = 0.0
    for v in arr:
        if v > peak: peak = v
        dd = max(dd, (peak - v) / max(peak, 1e-9))
    return dd


# ─── Logging ──────────────────────────────────────────────────────────────────

def _print_header(pos_size: float, conf_floor: float, horizon: int):
    W = 68
    sep = "=" * W
    print(flush=True)
    print(sep, flush=True)
    print("  BACKTEST CONFIGURATION", flush=True)
    print(sep, flush=True)
    print(f"  Mode:              Long-only (no short selling)", flush=True)
    print(f"  Horizon:           {horizon} days per position", flush=True)
    print(f"  Position size:     {pos_size:.0%} of available cash per BUY", flush=True)
    print(f"  Confidence floor:  {conf_floor:.2f}", flush=True)
    print(f"  Brokerage:         {BROKERAGE_PCT:.2%} per leg", flush=True)
    print(f"  Slippage:          {SLIPPAGE_PCT:.2%} per leg  (round-trip: {COST_PER_LEG*2:.2%})", flush=True)
    print(f"  Min trade value:   ₹{MIN_TRADE_VALUE:,.0f}", flush=True)
    print(f"  Overlap:           PREVENTED  (one position at a time)", flush=True)
    print(f"  Sell rule:         Only when shares held > 0", flush=True)
    print(sep, flush=True)
    print(flush=True)

    # Trade log column headers
    print(
        f"  {\'Day\':<9}  {\'Action\':<8}  {\'Str\':>6}  "
        f"{\'Pred%\':>6}  {\'Conf\':>5}  "
        f"{\'Price\':>9}  {\'Shares\':>6}  "
        f"{\'NetPnL\':>10}  {\'Cash\':>10}  "
        f"{\'Total\':>10}  Result",
        flush=True,
    )
    print("  " + "-" * 95, flush=True)


def _print_results(r: dict, portfolio: Portfolio):
    """Print final summary — called exactly once, all output flush=True."""
    W   = 68
    sep = "=" * W
    print(flush=True)
    print(sep, flush=True)
    print("  BACKTEST RESULTS — PORTFOLIO SUMMARY", flush=True)
    print(sep, flush=True)

    # ── Capital & P&L ────────────────────────────────────────────────────
    print(f"  Starting Capital:     ₹{r[\'initial_capital\']:>12,.2f}", flush=True)
    print(f"  Final Portfolio Value:₹{r[\'final_value\']:>12,.2f}  "
          f"({r[\'total_return_pct\']:+.2f}%)", flush=True)
    print(f"    Cash (uninvested):  ₹{r[\'cash\']:>12,.2f}", flush=True)
    if r[\'shares_held\'] > 0:
        print(f"    Open position:       {r[\'shares_held\']:>6} shares held", flush=True)
        print(f"    Unrealised P&L:    ₹{r[\'unrealised_pnl\']:>+12,.2f}  "
              "(included in final value)", flush=True)
    print(f"  Realised P&L:         ₹{r[\'realised_pnl\']:>+12,.2f}  "
          f"(from {r[\'n_trades\']} closed trades)", flush=True)
    print(f"  Transaction Costs:    ₹{r[\'total_costs\']:>12,.2f}  "
          f"({r[\'total_costs\']/r[\'initial_capital\']*100:.2f}% of capital)", flush=True)

    print(flush=True)

    # ── Trade statistics ──────────────────────────────────────────────────
    print(f"  ── Trade Statistics ───────────────────────────────────────", flush=True)
    print(f"  Total Days:           {r[\'n_days\']:>6}", flush=True)
    print(f"  Closed Trades:        {r[\'n_trades\']:>6}", flush=True)
    if r[\'n_trades\'] > 0:
        acc = r[\'accuracy\']
        print(f"  Win Rate:             {acc:>6.2%}  "
              f"({r[\'wins\']} wins / {r[\'losses\']} losses)", flush=True)
        print(f"  Avg Win:             ₹{r[\'avg_win_inr\']:>+12,.2f}", flush=True)
        print(f"  Avg Loss:            ₹{r[\'avg_loss_inr\']:>+12,.2f}", flush=True)
        pf = r[\'profit_factor\']
        pf_note = ("(>1.5=good)" if pf>=1.5 else
                   ("(>1.0=profitable)" if pf>=1.0 else "(loss-making)"))
        print(f"  Profit Factor:        {pf:>6.3f}  {pf_note}", flush=True)
        print(f"  Best Win Streak:      {r[\'best_streak\']:>6}", flush=True)
        print(f"  Worst Loss Streak:    {r[\'worst_streak\']:>6}", flush=True)

    print(flush=True)

    # ── Risk ──────────────────────────────────────────────────────────────
    print(f"  ── Risk Metrics ───────────────────────────────────────────", flush=True)
    sr = r[\'sharpe_ratio\']
    sr_note = ("(>1.5=strong)" if sr>1.5 else
               ("(>1.0=acceptable)" if sr>1.0 else
                ("(>0=marginal)" if sr>0 else "(negative=losing)")))
    print(f"  Sharpe Ratio:         {sr:>6.3f}  {sr_note}", flush=True)
    dd = r[\'max_drawdown_pct\']
    dd_note = "(<15%=good)" if dd<15 else ("(<30%=OK)" if dd<30 else "(>30%=high risk)")
    print(f"  Max Drawdown:         {dd:>6.2f}%  {dd_note}", flush=True)

    print(flush=True)

    # ── Strength breakdown ────────────────────────────────────────────────
    if r[\'n_strong\'] or r[\'n_medium\']:
        print(f"  ── By Signal Strength ─────────────────────────────────────", flush=True)
        if r[\'n_strong\']:
            print(f"  STRONG trades:        {r[\'n_strong\']:>6}  "
                  f"acc={r[\'strong_accuracy\']:.2%}", flush=True)
        if r[\'n_medium\']:
            print(f"  MEDIUM trades:        {r[\'n_medium\']:>6}  "
                  f"acc={r[\'medium_accuracy\']:.2%}", flush=True)
        print(flush=True)

    # ── Interpretation ────────────────────────────────────────────────────
    print(f"  ── Interpretation ─────────────────────────────────────────", flush=True)
    ret = r[\'total_return_pct\']
    if r[\'n_trades\'] == 0:
        print("  ⚠  ZERO TRADES — predictions all below threshold.", flush=True)
        print("     Retrain V4 model, or lower --confidence threshold.", flush=True)
    elif ret > 200:
        print("  ⚠  HIGH RETURNS — verify no data leakage in feature pipeline.", flush=True)
        print("     Check: are features computed on future data accidentally?", flush=True)
    elif sr > 1.5 and r[\'accuracy\'] >= 0.57:
        print("  ✓  STRONG — Sharpe > 1.5 with good accuracy.", flush=True)
        print("     Paper-trade for 30+ days before using real capital.", flush=True)
    elif sr > 0.8:
        print("  ~  ACCEPTABLE — Sharpe > 0.8, model has some edge.", flush=True)
    else:
        print("  ✗  WEAK — Sharpe < 0.8. Model not ready for live trading.", flush=True)
        print("     Try: more stocks in pretrain, start_date 2010, retrain.", flush=True)

    print(sep, flush=True)
    print(flush=True)


def _save_csv(trades: list, path: str):
    if not trades:
        print(f"  No trades to save.", flush=True)
        return
    fields = [
        "trade_num", "open_day_idx", "close_day_idx",
        "signal", "strength", "pred_return_pct", "confidence",
        "entry_price", "exit_price", "shares",
        "gross_pnl", "costs", "net_pnl",
        "is_win", "cash_before", "cash_after", "portfolio_value_after",
    ]
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for t in trades:
            w.writerow({
                "trade_num":             t.trade_num,
                "open_day_idx":          t.open_day_idx,
                "close_day_idx":         t.close_day_idx,
                "signal":                t.signal,
                "strength":              t.strength,
                "pred_return_pct":       round(t.pred_return * 100, 4),
                "confidence":            round(t.confidence, 4),
                "entry_price":           round(t.entry_price, 2),
                "exit_price":            round(t.exit_price, 2),
                "shares":                t.shares,
                "gross_pnl":             t.gross_pnl,
                "costs":                 t.costs,
                "net_pnl":               t.net_pnl,
                "is_win":                t.is_win,
                "cash_before":           t.cash_before,
                "cash_after":            t.cash_after,
                "portfolio_value_after": t.portfolio_value_after,
            })


# ─── CLI ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Portfolio-based backtest for StockForecastNet V5",
        formatter_class=argparse.RawTextHelpFormatter,
        epilog=(
            "Examples:\n"
            "  python backtest_v2.py --data data/RELIANCE/RELIANCE_daily_2010-01-01_2026-04-09.parquet\n"
            "  python backtest_v2.py --data ... --log_trades --csv trades.csv\n"
            "  python backtest_v2.py --data ... --position_size 0.20 --confidence 0.55\n"
        ),
    )
    parser.add_argument("--data",          required=True)
    parser.add_argument("--model",         default="model_v2.pth")
    parser.add_argument("--config",        default="model_v2_config.pth")
    parser.add_argument("--scaler",        default="scaler_v2.pkl")
    parser.add_argument("--confidence",    type=float, default=CONFIDENCE_FLOOR)
    parser.add_argument("--position_size", type=float, default=0.20,
                        help="Fraction of cash per BUY (default 0.20 = 20%%)")
    parser.add_argument("--horizon",       type=int,   default=3)
    parser.add_argument("--window",        type=int,   default=30)
    parser.add_argument("--device",        default="cpu",
                        choices=["cpu", "cuda", "mps"])
    parser.add_argument("--log_trades",    action="store_true",
                        help="Print each buy/sell as it executes")
    parser.add_argument("--log_interval",  type=int,   default=250,
                        help="Print portfolio status every N days (0=off)")
    parser.add_argument("--csv",           default=None,
                        help="Save trade log to CSV file")
    args = parser.parse_args()

    import pandas as pd

    print(f"\nLoading: {args.data}", flush=True)
    df_raw = (pd.read_parquet(args.data) if args.data.endswith(".parquet")
              else pd.read_csv(args.data))

    # IMPORTANT: keep \'close\' in df for share-price tracking
    # add_features_v2 works on a df with \'close\'; it drops raw prices from
    # the returned FEATURE_COLS — but we preserve \'close\' via _close_prices
    # stored inside StockDatasetV2 (see dataset_v2.py)
    df = add_features_v2(df_raw)

    # Re-attach close prices from raw df to the feature df
    # (add_features_v2 drops raw prices from output — we re-add for backtest)
    df_raw_aligned = df_raw.reset_index(drop=True)
    df_with_close  = df.copy()
    # Map clean df rows back to original close prices
    # Since add_features_v2 drops NaN rows and resets index, we need
    # to carry close prices through. We do this by using df_raw\'s close
    # aligned to the same row count after NaN removal.
    # The cleanest approach: pass df_raw close prices alongside features.
    # We achieve this by ensuring the df passed to StockDatasetV2 has \'close\'.
    if "close" not in df.columns:
        # Reconstruct: add_features_v2 resets index after dropping NaN.
        # The raw df may be longer. We pass close from df_raw indexed to
        # the surviving rows. Use the fact that add_features_v2 prints
        # "N rows → M clean rows" — M rows survive.
        n_raw = len(df_raw)
        n_clean = len(df)
        # Simple heuristic: last n_clean rows of df_raw (warmup rows dropped from front)
        close_aligned = df_raw["close"].values[-n_clean:] if n_raw >= n_clean else df_raw["close"].values
        df["close"] = close_aligned[:n_clean]

    scaler  = joblib.load(args.scaler)
    dataset = StockDatasetV2(
        df, window=args.window, horizon=args.horizon, scaler=scaler
    )
    dataset.summary()
    print(flush=True)

    if os.path.exists(args.config):
        cfg = torch.load(args.config, map_location="cpu")
        print(f"  Config: {cfg}", flush=True)
    else:
        print(f"  [WARNING] Config not found, using defaults.", flush=True)
        cfg = {
            "input_dim": len(FEATURE_COLS), "window": args.window,
            "d_model": 64, "n_layers": 2, "n_heads": 4,
            "d_ff": 128, "dropout": 0.0, "horizon": args.horizon,
        }

    if cfg.get("input_dim") != dataset.n_features:
        cfg["input_dim"] = dataset.n_features

    model = StockForecastNet(**cfg)
    state = torch.load(args.model, map_location="cpu")
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f"  [INFO] {len(missing)} missing keys zero-initialised", flush=True)
    print(f"  {model}", flush=True)

    backtest_v2(
        model             = model,
        dataset           = dataset,
        horizon           = args.horizon,
        min_confidence    = args.confidence,
        position_size_pct = args.position_size,
        device            = args.device,
        log_trades        = args.log_trades,
        log_interval      = args.log_interval,
        csv_path          = args.csv,
    )

'''
(PROJECT_ROOT / "backtest_v2.py").write_text(_backtest_source)
print(f"✓ backtest_v2.py written ({len(_backtest_source):,} chars)")


In [ ]:
# ─── 3G: Import all modules ───────────────────────────────────────────────────
# Fresh imports now that all .py files are written to disk
import importlib, sys

# Remove stale cached modules if re-running cell
for mod_name in ["model_v2","features_v2","dataset_v2","utils","utils.trading_v2",
                  "backtest_v2","strategies","strategies.base"]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

from model_v2    import StockForecastNet
from features_v2 import add_features_v2, FEATURE_COLS
from dataset_v2  import StockDatasetV2, extract_time_features
from utils.trading_v2 import generate_signal_v2, pred_to_confidence, CONFIDENCE_FLOOR

print("✓ All modules imported successfully:")
print(f"  StockForecastNet  — PatchTST + ReVIN + CI Transformer")
print(f"  add_features_v2   — {len(FEATURE_COLS)} stationary features from 19 strategies")
print(f"  StockDatasetV2    — sliding window, multi-step labels, time features")
print(f"  generate_signal_v2 — BUY/SELL/HOLD signal logic")


## 📊 Section 4 — Data Fetching

In [ ]:
# ─── 4A: Fetch NSE data via yfinance ─────────────────────────────────────────
# yfinance is free, no API key, works directly on Colab.
# NSE ticker format: SYMBOL.NS  e.g. TCS.NS, RELIANCE.NS
# Alternative: upload your own .parquet files from local machine to /content/

import yfinance as yf
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

def fetch_nse_stock(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Fetch NSE daily OHLCV from Yahoo Finance.
    Returns DataFrame with lowercase columns: datetime, open, high, low, close, volume
    Column format matches what add_features_v2() expects.
    """
    print(f"  Fetching {ticker:<20}", end=" ", flush=True)
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)

    if df.empty:
        print("FAILED — no data returned")
        return None

    # yfinance sometimes returns MultiIndex columns — flatten them
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0].lower() for col in df.columns]
    else:
        df.columns = [c.lower() for c in df.columns]

    # Rename Date index to datetime column
    df = df.reset_index()
    date_col = "date" if "date" in df.columns else df.columns[0]
    df = df.rename(columns={date_col: "datetime"})
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Keep only needed columns, drop NaN, sort oldest-first
    df = df[["datetime", "open", "high", "low", "close", "volume"]].dropna()
    df = df.sort_values("datetime").reset_index(drop=True)

    span = f"{df['datetime'].min().date()} → {df['datetime'].max().date()}"
    print(f"✓ {len(df):>5,} rows  [{span}]")
    return df

print(f"Fetching {len(SYMBOLS)} stocks from Yahoo Finance...")
print(f"Period: {START_DATE} → {END_DATE}")
print()

RAW_DATA = {}   # ticker → raw DataFrame (before feature engineering)
for ticker in SYMBOLS:
    df = fetch_nse_stock(ticker, START_DATE, END_DATE)
    if df is not None and len(df) >= 500:
        RAW_DATA[ticker] = df
        # Save parquet to data/ folder (matches local project structure)
        sym_clean = ticker.replace(".NS", "")
        out_path = PROJECT_ROOT / "data" / f"{sym_clean}_daily.parquet"
        df.to_parquet(out_path, index=False)
    elif df is not None:
        print(f"  ⚠ SKIP {ticker}: only {len(df)} rows (need ≥500)")

print()
print(f"✓ {len(RAW_DATA)} stocks ready for feature engineering")
if len(RAW_DATA) < 3:
    print("  ⚠ WARNING: Very few stocks. Consider adding more to SYMBOLS in Section 2.")


In [ ]:
# ─── 4B: Upload local .parquet files (alternative to yfinance) ───────────────
# If you already have parquet files from Upstox on your local machine,
# run this cell to upload them instead of using yfinance.
# Skip this cell if you used 4A.

UPLOAD_FROM_LOCAL = False  # ← Set True to upload your own parquet files

if UPLOAD_FROM_LOCAL:
    from google.colab import files
    print("Select your .parquet files to upload (hold Ctrl/Cmd for multiple)...")
    uploaded = files.upload()

    for fname, content in uploaded.items():
        # Infer ticker from filename e.g. RELIANCE_daily_2010-01-01_2026-04-09.parquet
        ticker_raw = fname.split("_")[0].upper()
        path = PROJECT_ROOT / "data" / fname
        path.write_bytes(content)

        df = pd.read_parquet(path)
        # Standardise column names to lowercase
        df.columns = [c.lower() for c in df.columns]
        if "datetime" not in df.columns and df.index.name in ("datetime","date","Date"):
            df = df.reset_index().rename(columns={"date":"datetime","Date":"datetime"})
        df["datetime"] = pd.to_datetime(df["datetime"])

        ticker_key = ticker_raw + ".NS"
        RAW_DATA[ticker_key] = df
        print(f"  ✓ Uploaded {fname}: {len(df):,} rows ({ticker_key})")
    print(f"\n✓ Total stocks loaded: {list(RAW_DATA.keys())}")


## 🔬 Section 5 — Feature Engineering & Dataset Construction

In [ ]:
# ─── 5A: Feature engineering ─────────────────────────────────────────────────
# add_features_v2 computes all 56 stationary technical features from raw OHLCV.
# This is the EXACT same function as in your local features_v2.py.
# Applies all 19 strategies, drops NaN warmup rows (~100 rows consumed).

import numpy as np
import warnings
warnings.filterwarnings("ignore")

FEATURED_DATA = {}   # ticker → featured DataFrame

print("Engineering 56 features for each stock...")
print("(19 strategies: MA, MACD, RSI, BB, Breakout, VWAP, ATR, Candlestick,")
print(" Momentum, Stochastic, CCI, Williams%R, OBV, Donchian, SuperTrend,")
print(" Keltner, Heikin-Ashi, Pivot, Ichimoku + calendar features)")
print()

for ticker, df_raw in RAW_DATA.items():
    print(f"  {ticker:<20}", end=" ", flush=True)
    try:
        df_feat = add_features_v2(df_raw.copy())
        FEATURED_DATA[ticker] = df_feat
        # add_features_v2 returns only FEATURE_COLS — re-attach close + datetime
        # for backtest price tracking
        n = len(df_feat)
        n_raw = len(df_raw)
        warmup = n_raw - n
        # Align: featured rows correspond to the last n rows of df_raw
        df_feat["close"]    = df_raw["close"].values[warmup : warmup + n]
        df_feat["datetime"] = df_raw["datetime"].values[warmup : warmup + n]
        print(f"✓ {n:,} rows | {len(df_feat.columns)} cols")
    except Exception as e:
        print(f"✗ FAILED: {e}")
        import traceback; traceback.print_exc()

print(f"\n✓ {len(FEATURED_DATA)} stocks ready: {list(FEATURED_DATA.keys())}")
print(f"  FEATURE_COLS: {len(FEATURE_COLS)} features")
print(f"  Sample feature names: {FEATURE_COLS[:5]} ... {FEATURE_COLS[-3:]}")


In [ ]:
# ─── 5B: Build train / validation datasets ───────────────────────────────────
# For each stock:
#   - Chronological split: first 80% → train, last 20% → val
#   - 10-day GAP between train and val (prevents autocorrelation leakage)
# A SHARED RobustScaler is fitted on ALL training data combined.
# ReVIN in the model handles per-window normalization on top of this.

from sklearn.preprocessing import RobustScaler
from torch.utils.data import ConcatDataset
import joblib

print(f"Building datasets: seq_len={SEQ_LEN}, horizon={HORIZON}d,")
print(f"  noise_threshold={NOISE_THRESHOLD}, val_split={VAL_SPLIT}, gap={GAP}")
print()

# Step 1: Chronological split
train_dfs, val_dfs = {}, {}
for ticker, df in FEATURED_DATA.items():
    n      = len(df)
    n_val  = int(n * VAL_SPLIT)
    n_tr   = n - n_val - GAP
    if n_tr < SEQ_LEN + HORIZON + 100:
        print(f"  ⚠ SKIP {ticker}: only {n_tr} training rows after split")
        continue
    train_dfs[ticker] = df.iloc[:n_tr].copy()
    val_dfs[ticker]   = df.iloc[n_tr + GAP:].copy()
    print(f"  {ticker:<20}: {n_tr:,} train + {n_val:,} val rows")

if not train_dfs:
    raise RuntimeError("No usable training data. Check SYMBOLS and date range.")

# Step 2: Fit ONE shared RobustScaler on ALL training features
print("\nFitting shared RobustScaler on combined training data...")
combined_arr = np.vstack([df[FEATURE_COLS].values for df in train_dfs.values()])
SHARED_SCALER = RobustScaler()
SHARED_SCALER.fit(combined_arr)
print(f"  Fitted on {len(combined_arr):,} rows from {len(train_dfs)} stocks")

# Step 3: Build PyTorch datasets
TRAIN_SETS, VAL_SETS = [], []

print("\nBuilding PyTorch datasets...")
for ticker, df in train_dfs.items():
    try:
        ds = StockDatasetV2(
            df, window=SEQ_LEN, horizon=HORIZON,
            noise_threshold=NOISE_THRESHOLD,
            scaler=SHARED_SCALER, symbol=ticker,
        )
        ds.summary()
        TRAIN_SETS.append(ds)
    except ValueError as e:
        print(f"  ⚠ SKIP train {ticker}: {e}")

for ticker, df in val_dfs.items():
    try:
        ds = StockDatasetV2(
            df, window=SEQ_LEN, horizon=HORIZON,
            noise_threshold=NOISE_THRESHOLD,
            scaler=SHARED_SCALER, symbol=ticker,
        )
        VAL_SETS.append(ds)
    except ValueError as e:
        print(f"  ⚠ SKIP val {ticker}: {e}")

if not TRAIN_SETS:
    raise RuntimeError("All datasets failed. Check data quality.")

TRAIN_DS = ConcatDataset(TRAIN_SETS) if len(TRAIN_SETS) > 1 else TRAIN_SETS[0]
VAL_DS   = ConcatDataset(VAL_SETS)   if len(VAL_SETS)   > 1 else VAL_SETS[0]

n_tr_total = sum(len(d) for d in TRAIN_SETS)
n_va_total = sum(len(d) for d in VAL_SETS)
eff_ratio  = n_tr_total * len(FEATURE_COLS) / 270579  # approx param count

print(f"\n{'='*55}")
print(f"  {n_tr_total:,} train + {n_va_total:,} val samples")
print(f"  Stocks: {[d.symbol for d in TRAIN_SETS]}")
print(f"  Effective CI ratio: {n_tr_total}×{len(FEATURE_COLS)}/270K = {eff_ratio:.1f}x")
if eff_ratio < 2:
    print("  ⚠ LOW — add more diverse stocks for better accuracy")
elif eff_ratio < 5:
    print("  ~ MARGINAL — expect 53-58% accuracy")
else:
    print("  ✓ GOOD — expect 57-64% accuracy")
print(f"{'='*55}")


## 🚀 Section 6 — Model Training

In [ ]:
# ─── 6A: Build model + detect device ─────────────────────────────────────────
import torch
import random, numpy as np

# Reproducibility
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"  {p.name}  |  {p.total_memory/1024**3:.1f} GB VRAM")
    torch.backends.cudnn.benchmark = True
else:
    print("  ⚠ CPU — training will be much slower")

# Build model
MODEL = StockForecastNet(
    n_features  = len(FEATURE_COLS),
    seq_len     = SEQ_LEN,
    horizon     = HORIZON,
    patch_size  = PATCH_SIZE,
    stride      = STRIDE,
    d_model     = D_MODEL,
    n_heads     = N_HEADS,
    n_layers    = N_LAYERS,
    d_ff        = D_FF,
    dropout     = DROPOUT,
).to(DEVICE)

print(f"\n{MODEL}")
c = MODEL.count_parameters()
print(f"\n  Parameter breakdown:")
print(f"  {'Component':<28} {'Params':>10}")
print(f"  {'-'*40}")
for k, v in c.items():
    if k not in ("total", "size_mb"):
        print(f"  {k:<28} {v:>10,}")
print(f"  {'─'*40}")
print(f"  {'TOTAL':<28} {c['total']:>10,}  ({c['size_mb']} MB)")
B_C = BATCH_SIZE * len(FEATURE_COLS)
n_p = (SEQ_LEN - PATCH_SIZE) // STRIDE + 1
fmb = B_C * n_p * D_MODEL * 4 / 1024 / 1024
print(f"\n  B×C = {BATCH_SIZE}×{len(FEATURE_COLS)} = {B_C} | fused ≈ {fmb:.1f} MB")


In [ ]:
# ─── 6B: Training loop definition ────────────────────────────────────────────
# Uses _iter_batches() — pure Python manual batch iteration.
# NO DataLoader, NO WeightedRandomSampler.
# Root cause of Windows crash: WeightedRandomSampler calls torch.multinomial()
# which triggers OS-level thread kill on Windows+Intel MKL, bypassing all
# Python exception handling. This function avoids that entirely.

import torch.nn as nn
import time

def _iter_batches(dataset, batch_size, shuffle, device):
    """Manual batch iterator. No DataLoader threading. 100% portable."""
    n = len(dataset)
    idx = torch.randperm(n).tolist() if shuffle else list(range(n))
    for start in range(0, n - batch_size + 1, batch_size):
        batch = idx[start : start + batch_size]
        Xs, tfs, ys = [], [], []
        for i in batch:
            x, tf, y = dataset[i]
            Xs.append(x); tfs.append(tf); ys.append(y)
        yield (torch.stack(Xs).to(device),
               torch.stack(tfs).to(device),
               torch.stack(ys).to(device))

def train_stockforecastnet(model, train_ds, val_ds, device,
                            batch_size=BATCH_SIZE, epochs=EPOCHS,
                            lr=LR, patience=PATIENCE, horizon=HORIZON,
                            weight_decay=WEIGHT_DECAY):
    """
    Full training loop for StockForecastNet V5.

    Loss: multi-step HuberLoss averaged over all horizon steps.
    Metric: direction accuracy — sign(pred[:,-1]) == sign(y[:,-1])
    Schedule: CosineAnnealingWarmRestarts (restart at epoch 20, then 60, 140)
    patience=30 ensures early stopping doesn't trigger before the epoch-20 restart.
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=weight_decay, betas=(0.9, 0.999))
    # T_0=20: first LR restart at epoch 20
    # patience > 20 ensures training survives to see the restart
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6)
    loss_fn   = nn.HuberLoss(delta=0.02)
    n_tr      = max(len(train_ds) // batch_size, 1)
    n_va      = max(len(val_ds)   // batch_size, 1)

    best_loss, best_acc, best_state, no_improve = float("inf"), 0.0, None, 0

    # Dry run to catch dimension errors before epoch 1
    print("  Dry-run: validating forward pass...", flush=True)
    model.eval()
    with torch.no_grad():
        x0, tf0, y0 = train_ds[0]
        out = model(x0.unsqueeze(0).to(device), tf0.unsqueeze(0).to(device))
    print(f"  ✓ input {tuple(x0.shape)} → output {tuple(out.shape)}", flush=True)
    del x0, tf0, y0, out

    print(f"\n  {'Ep':>4}  {'TrLoss':>9}  {'TrAcc':>6}  {'VaLoss':>9}  {'VaAcc':>6}  {'LR':>9}  {'s/ep':>6}", flush=True)
    print("  " + "─" * 62, flush=True)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss = tr_correct = tr_total = 0

        for X, tf, y in _iter_batches(train_ds, batch_size, shuffle=True, device=device):
            pred  = model(X, tf)
            loss  = sum(loss_fn(pred[:,h], y[:,h]) for h in range(horizon)) / horizon
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss    += loss.item()
            tr_correct += int((torch.sign(pred[:,-1]) == torch.sign(y[:,-1])).sum())
            tr_total   += y.size(0)

        scheduler.step()

        model.eval()
        va_loss = va_correct = va_total = 0
        with torch.no_grad():
            for X, tf, y in _iter_batches(val_ds, batch_size, shuffle=False, device=device):
                pred    = model(X, tf)
                va_loss += (sum(loss_fn(pred[:,h],y[:,h]) for h in range(horizon))/horizon).item()
                va_correct += int((torch.sign(pred[:,-1]) == torch.sign(y[:,-1])).sum())
                va_total   += y.size(0)

        tr_acc  = tr_correct / max(tr_total, 1)
        va_acc  = va_correct / max(va_total, 1)
        avg_tr  = tr_loss / n_tr
        avg_va  = va_loss / n_va
        cur_lr  = optimizer.param_groups[0]["lr"]
        elapsed = time.time() - t0

        print(f"  {epoch:>4}  {avg_tr:>9.5f}  {tr_acc:>6.3f}  "
              f"{avg_va:>9.5f}  {va_acc:>6.3f}  {cur_lr:>9.2e}  {elapsed:>6.1f}s", flush=True)
        if epoch == 1:
            eta = elapsed * epochs / 60
            print(f"  [ETA: ~{eta:.0f} min for {epochs} epochs — actual will be less with early stop]", flush=True)

        if avg_va < best_loss:
            best_loss, best_acc, no_improve = avg_va, va_acc, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n  Early stop at epoch {epoch} (patience={patience}).", flush=True)
                print(f"  Best val loss={best_loss:.5f}  val acc={best_acc:.2%}", flush=True)
                break

    return best_state, best_acc

print("✓ Training function defined")
print("  Features: multi-step HuberLoss, direction accuracy, CosineAnnealingWarmRestarts")
print("  Dry-run validation, manual batch iteration (no DataLoader crash)")


In [ ]:
# ─── 6C: ▶ RUN TRAINING ──────────────────────────────────────────────────────
import time

print("=" * 60)
print("  TRAINING StockForecastNet V5")
print("=" * 60)
print(f"  Device:      {DEVICE}")
print(f"  Train size:  {len(TRAIN_DS):,} samples")
print(f"  Val size:    {len(VAL_DS):,} samples")
print(f"  Batch:       {BATCH_SIZE}  |  Epochs: {EPOCHS}  |  Patience: {PATIENCE}")
print(f"  Horizon:     {HORIZON}d  |  seq_len: {SEQ_LEN}  |  LR: {LR}")
print("=" * 60)
print()

t_start = time.time()
BEST_STATE, BEST_ACC = train_stockforecastnet(
    model        = MODEL,
    train_ds     = TRAIN_DS,
    val_ds       = VAL_DS,
    device       = DEVICE,
    batch_size   = BATCH_SIZE,
    epochs       = EPOCHS,
    lr           = LR,
    patience     = PATIENCE,
    horizon      = HORIZON,
    weight_decay = WEIGHT_DECAY,
)
TOTAL_MINS = (time.time() - t_start) / 60

print(f"\n{'='*55}")
print(f"  Training complete in {TOTAL_MINS:.1f} minutes")
print(f"  Best validation accuracy: {BEST_ACC:.2%}")
if BEST_ACC >= 0.60:
    print("  ✓ STRONG — model ready for backtesting and deployment")
elif BEST_ACC >= 0.56:
    print("  ~ GOOD — acceptable for conservative use")
elif BEST_ACC >= 0.53:
    print("  ~ MARGINAL — add more diverse stocks for better accuracy")
else:
    print("  ✗ WEAK — use diverse cross-sector stocks (see Section 2)")
print(f"{'='*55}")


## 📊 Section 7 — Evaluation & Validation Analysis

In [ ]:
# ─── 7A: Load best weights + evaluate on validation set ──────────────────────
import math
import torch
import numpy as np

# Load best weights found during training
if BEST_STATE is not None:
    MODEL.load_state_dict(BEST_STATE)
    print(f"✓ Best weights loaded (val acc: {BEST_ACC:.2%})")

MODEL.eval()
DEVICE_CPU = torch.device("cpu")
MODEL_CPU = MODEL.to(DEVICE_CPU)

# Full pass over validation set
all_preds, all_labels, all_confs = [], [], []
with torch.no_grad():
    for val_ds in VAL_SETS:
        for i in range(len(val_ds)):
            x, tf, y = val_ds[i]
            pred = MODEL_CPU(x.unsqueeze(0), tf.unsqueeze(0))
            pred_val = pred[0, -1].item()
            all_preds.append(pred_val)
            all_labels.append(y[-1].item())
            all_confs.append(1.0 / (1.0 + math.exp(-abs(pred_val) * 100)))

preds  = np.array(all_preds)
labels = np.array(all_labels)
confs  = np.array(all_confs)

# Direction accuracy
dir_correct = (np.sign(preds) == np.sign(labels))
dir_acc = dir_correct.mean()

# Signal distribution
signals = []
for p, c in zip(preds, confs):
    d = 1 if p > 0 else 0
    sig, strength = generate_signal_v2(d, c, p)
    signals.append((sig, strength))

buy_strong  = sum(1 for s,st in signals if s=="BUY"  and st=="STRONG")
buy_medium  = sum(1 for s,st in signals if s=="BUY"  and st=="MEDIUM")
sell_strong = sum(1 for s,st in signals if s=="SELL" and st=="STRONG")
sell_medium = sum(1 for s,st in signals if s=="SELL" and st=="MEDIUM")
hold        = sum(1 for s,st in signals if s=="HOLD")
n_total     = len(signals)

print("=" * 55)
print("  VALIDATION EVALUATION RESULTS")
print("=" * 55)
print(f"  Total val samples:   {n_total:,}")
print(f"  Direction accuracy:  {dir_acc:.2%}  ({dir_correct.sum()} correct)")
print(f"  Mean |prediction|:   {np.abs(preds).mean():.4f}  ({np.abs(preds).mean()*100:.2f}%)")
print(f"  Mean confidence:     {confs.mean():.3f}")
print()
print(f"  Signal distribution:")
print(f"    BUY  STRONG: {buy_strong:>6,}  ({buy_strong/n_total:.1%})")
print(f"    BUY  MEDIUM: {buy_medium:>6,}  ({buy_medium/n_total:.1%})")
print(f"    SELL STRONG: {sell_strong:>6,}  ({sell_strong/n_total:.1%})")
print(f"    SELL MEDIUM: {sell_medium:>6,}  ({sell_medium/n_total:.1%})")
print(f"    HOLD:        {hold:>6,}  ({hold/n_total:.1%})")
print()
if (buy_strong + buy_medium + sell_strong + sell_medium) == 0:
    print("  ⚠ No signals above threshold — model may need retraining")
    print(f"    Current CONFIDENCE_FLOOR = {CONFIDENCE_FLOOR}")
    print(f"    Try lower confidence: generate_signal_v2 threshold = 0.50")
else:
    trade_rate = (n_total - hold) / n_total
    print(f"  Trade rate: {trade_rate:.1%} of days would generate a signal")
print("=" * 55)


In [ ]:
# ─── 7B: Per-stock accuracy breakdown ────────────────────────────────────────
print("Per-stock validation accuracy:")
print()

for val_ds in VAL_SETS:
    n = len(val_ds)
    correct = 0
    strong_correct = strong_total = 0

    for i in range(n):
        x, tf, y = val_ds[i]
        with torch.no_grad():
            pred = MODEL_CPU(x.unsqueeze(0), tf.unsqueeze(0))
        p = pred[0, -1].item()
        c = 1.0 / (1.0 + math.exp(-abs(p) * 100))
        d = 1 if p > 0 else 0
        sig, strength = generate_signal_v2(d, c, p)
        if np.sign(p) == np.sign(y[-1].item()):
            correct += 1
        if strength == "STRONG":
            strong_total += 1
            if np.sign(p) == np.sign(y[-1].item()):
                strong_correct += 1

    acc = correct / n
    strong_acc = strong_correct / strong_total if strong_total > 0 else 0
    bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"  {val_ds.symbol:<20} [{bar}] {acc:.2%}  "
          f"(STRONG: {strong_correct}/{strong_total} = {strong_acc:.1%})")

print()
print("Note: STRONG signals (higher confidence) should have higher accuracy")
print("than overall accuracy. If not, model needs more data or training.")


## 📈 Section 8 — Portfolio Backtesting

In [ ]:
# ─── 8A: Backtest on a single stock ──────────────────────────────────────────
# Runs the EXACT same backtest_v2.py logic — share quantities, cost tracking,
# unrealised P&L, long-only portfolio. No simplification.
# This tests the model on the full date range of available data.

import sys
sys.path.insert(0, str(PROJECT_ROOT))

from backtest_v2 import backtest_v2, Position, ClosedTrade, Portfolio

# ─── Choose which stock to backtest ──────────────────────────────────────────
BACKTEST_TICKER = list(RAW_DATA.keys())[0]  # first stock by default
# Change to any other ticker in your SYMBOLS list, e.g.:
# BACKTEST_TICKER = "RELIANCE.NS"

print(f"Backtesting on: {BACKTEST_TICKER}")
print(f"Full date range: {RAW_DATA[BACKTEST_TICKER]['datetime'].min().date()} "
      f"→ {RAW_DATA[BACKTEST_TICKER]['datetime'].max().date()}")

# Build dataset for backtesting (uses full data, no train/val split)
df_bt_raw  = RAW_DATA[BACKTEST_TICKER].copy()
df_bt_feat = FEATURED_DATA[BACKTEST_TICKER].copy()

# Ensure close prices are attached for share-quantity tracking
if "_close_prices" not in dir(None):
    pass  # will be set in StockDatasetV2

bt_dataset = StockDatasetV2(
    df_bt_feat,
    window          = SEQ_LEN,
    horizon         = HORIZON,
    noise_threshold = 0.0,      # keep ALL samples for backtest (don't filter)
    scaler          = SHARED_SCALER,
    symbol          = BACKTEST_TICKER,
)
bt_dataset.summary()

print()
# Run backtest
BT_RESULTS = backtest_v2(
    model             = MODEL_CPU,
    dataset           = bt_dataset,
    horizon           = HORIZON,
    min_confidence    = CONFIDENCE_FLOOR,
    position_size_pct = 0.20,   # 20% of cash per trade
    device            = "cpu",
    log_trades        = False,  # set True for verbose trade log
    log_interval      = 500,    # print progress every 500 days
    csv_path          = str(PROJECT_ROOT / "exports" / "backtest_trades.csv"),
)


In [ ]:
# ─── 8B: Backtest on multiple stocks ─────────────────────────────────────────
print("\n" + "="*55)
print("  BACKTEST SUMMARY ACROSS ALL STOCKS")
print("="*55)
print(f"  {'Ticker':<22} {'Return%':>8} {'Win%':>7} {'Trades':>7} {'Sharpe':>8}")
print("  " + "─" * 56)

all_results = {}
for ticker in list(FEATURED_DATA.keys())[:5]:  # limit to first 5 for speed
    df_feat = FEATURED_DATA[ticker].copy()
    try:
        ds = StockDatasetV2(
            df_feat, window=SEQ_LEN, horizon=HORIZON,
            noise_threshold=0.0, scaler=SHARED_SCALER, symbol=ticker,
        )
        r = backtest_v2(
            model=MODEL_CPU, dataset=ds, horizon=HORIZON,
            min_confidence=CONFIDENCE_FLOOR, position_size_pct=0.20,
            device="cpu", log_trades=False, log_interval=0,
        )
        all_results[ticker] = r
        ret_pct = r['total_return_pct']
        win_pct = r['accuracy'] * 100
        n_tr    = r['n_trades']
        sharpe  = r['sharpe_ratio']
        marker  = "✓" if ret_pct > 0 else "✗"
        print(f"  {marker} {ticker:<20} {ret_pct:>+8.1f}% {win_pct:>6.1f}% "
              f"{n_tr:>7} {sharpe:>8.2f}")
    except Exception as e:
        print(f"  ✗ {ticker:<20} FAILED: {e}")

print("  " + "─" * 56)
print()
print("Note: Backtest uses full data range (train + val combined).")
print("For unbiased backtest, run only on val period dates.")
print(f"\nTrade log saved to: {PROJECT_ROOT / 'exports' / 'backtest_trades.csv'}")


## 💾 Section 9 — Export Model for FastAPI / AWS Deployment

In [ ]:
# ─── 9A: Save model artifacts ────────────────────────────────────────────────
# Saves the 3 files required by your local FastAPI (api_v2.py):
#   pretrained_v5.pth        → model weights
#   pretrained_v5_config.pth → get_config() dict (architecture spec)
#   scaler_v2.pkl            → fitted RobustScaler

import joblib, torch

# Ensure best weights are loaded
if BEST_STATE is not None:
    MODEL.load_state_dict(BEST_STATE)

MODELS_DIR.mkdir(exist_ok=True)
EXPORTS_DIR.mkdir(exist_ok=True)

weights_path = MODELS_DIR / WEIGHTS_FILE
config_path  = MODELS_DIR / CONFIG_FILE
scaler_path  = MODELS_DIR / SCALER_FILE

torch.save(BEST_STATE or MODEL.state_dict(), weights_path)
torch.save(MODEL.get_config(),               config_path)
joblib.dump(SHARED_SCALER,                   scaler_path)

print("✓ Model artifacts saved:")
print(f"  {weights_path}  ({weights_path.stat().st_size/1024:.1f} KB)")
print(f"  {config_path}  ({config_path.stat().st_size/1024:.1f} KB)")
print(f"  {scaler_path}  ({scaler_path.stat().st_size/1024:.1f} KB)")

# Verify: reload from disk and do a test forward pass
print("\nVerifying artifacts (reload from disk)...")
cfg_loaded   = torch.load(config_path, map_location="cpu")
model_reload = StockForecastNet(**cfg_loaded)
state_loaded = torch.load(weights_path, map_location="cpu")
missing, _   = model_reload.load_state_dict(state_loaded, strict=False)
model_reload.eval()

x0, tf0, _ = TRAIN_SETS[0][0]
with torch.no_grad():
    out = model_reload(x0.unsqueeze(0), tf0.unsqueeze(0))
print(f"  ✓ Reload OK — output shape: {tuple(out.shape)}  "
      f"primary pred: {out[0,-1].item():+.4f}")
del model_reload

# Print config for reference
print(f"\n  Architecture config saved:")
for k, v in cfg_loaded.items():
    print(f"    {k:<15}: {v}")


In [ ]:
# ─── 9B: Copy artifacts to Google Drive (if mounted) ─────────────────────────
import shutil

if DRIVE_SAVE_DIR is not None:
    for f in [weights_path, config_path, scaler_path]:
        dest = DRIVE_SAVE_DIR / f.name
        shutil.copy(f, dest)
        print(f"  ✓ Copied to Drive: {dest}")
    print(f"\n✓ All 3 artifacts saved to Drive: {DRIVE_SAVE_DIR}")
    print("  They will survive Colab session resets.")
else:
    print("Drive not mounted. Files are in /content/ai-trading-service/models/")
    print("Download them in the next cell before closing Colab.")


In [ ]:
# ─── 9C: Download artifacts to your local machine ────────────────────────────
# This triggers browser download dialogs.
# Run this cell to get the 3 files before closing Colab.

from google.colab import files as colab_files
import time

print("Downloading 3 artifacts to your computer...")
print("(3 browser dialogs will appear — check your downloads folder)")
print()

for fpath in [weights_path, config_path, scaler_path]:
    print(f"  Downloading {fpath.name}...", end=" ", flush=True)
    colab_files.download(str(fpath))
    print("done")
    time.sleep(0.8)

print()
print("=" * 55)
print("  DOWNLOAD COMPLETE")
print("=" * 55)
print()
print("Copy these 3 files to: apps/ai-trading-service/")
print(f"  {WEIGHTS_FILE}")
print(f"  {CONFIG_FILE}")
print(f"  {SCALER_FILE}")
print()
print("Then run locally:")
print("  python infer.py --symbol RELIANCE")
print("  python backtest_v2.py --data data/RELIANCE/...")
print()
print("For FastAPI (AWS EC2):")
print("  uvicorn api_v2:app --host 0.0.0.0 --port 8000")
print("  Environment variables needed:")
print("    MODEL_PATH=pretrained_v5.pth")
print("    CONFIG_PATH=pretrained_v5_config.pth")
print("    SCALER_PATH=scaler_v2.pkl")


## 🎯 Section 10 — Fine-tune on a Single Stock (Optional)

In [ ]:
# ─── 10A: Fine-tune pretrained model on one stock ─────────────────────────────
# After pretraining on multiple stocks, you can fine-tune on a specific stock
# with a lower learning rate for better per-stock performance.
# This cell is OPTIONAL — run if you want stock-specific fine-tuning.

FINETUNE = False       # ← Set True to run fine-tuning
FINETUNE_TICKER = list(FEATURED_DATA.keys())[0]   # stock to finetune on

if FINETUNE:
    print(f"Fine-tuning on: {FINETUNE_TICKER}")
    df_ft = FEATURED_DATA[FINETUNE_TICKER]
    n     = len(df_ft)
    n_val = int(n * VAL_SPLIT)
    n_tr  = n - n_val - GAP

    ft_train = StockDatasetV2(df_ft.iloc[:n_tr], window=SEQ_LEN, horizon=HORIZON,
                               noise_threshold=NOISE_THRESHOLD,
                               scaler=SHARED_SCALER, symbol=FINETUNE_TICKER)
    ft_val   = StockDatasetV2(df_ft.iloc[n_tr+GAP:], window=SEQ_LEN, horizon=HORIZON,
                               noise_threshold=NOISE_THRESHOLD,
                               scaler=SHARED_SCALER, symbol=FINETUNE_TICKER)

    # Reload best pretrained weights to start from
    MODEL.load_state_dict(BEST_STATE)

    print(f"  {len(ft_train):,} train + {len(ft_val):,} val samples")
    print(f"  LR: {LR/5:.2e}  (5x lower than pretrain)")
    print()

    ft_best_state, ft_acc = train_stockforecastnet(
        model=MODEL, train_ds=ft_train, val_ds=ft_val, device=DEVICE,
        batch_size=BATCH_SIZE, epochs=40, lr=LR/5, patience=15,
        horizon=HORIZON, weight_decay=WEIGHT_DECAY,
    )

    print(f"\n  Fine-tune val acc: {ft_acc:.2%}")
    if ft_acc > BEST_ACC:
        print(f"  ✓ Better than pretrain ({BEST_ACC:.2%}) — using fine-tuned weights")
        BEST_STATE = ft_best_state
        BEST_ACC   = ft_acc
    else:
        print(f"  ~ Pretrain ({BEST_ACC:.2%}) still better — keeping pretrained weights")
else:
    print("Fine-tuning skipped (FINETUNE=False)")
    print("Set FINETUNE=True and re-run this cell to fine-tune on a specific stock.")


## 🔍 Section 11 — Quick Inference Test

In [ ]:
# ─── 11: Test inference on the most recent available data ────────────────────
# Simulates what infer.py does locally.
# Uses the last SEQ_LEN days of the chosen stock to generate a signal.

import math

TEST_TICKER = list(FEATURED_DATA.keys())[0]
df_inf = FEATURED_DATA[TEST_TICKER]
n_rows = len(df_inf)

# Take the most recent SEQ_LEN rows
X_raw    = df_inf.tail(SEQ_LEN)[FEATURE_COLS].values
X_scaled = SHARED_SCALER.transform(X_raw)
X_t      = torch.tensor(X_scaled, dtype=torch.float32).unsqueeze(0)

# Time features
tf_arr = extract_time_features(df_inf, window_start=n_rows-SEQ_LEN, window_len=SEQ_LEN)
tf_t   = torch.tensor(tf_arr, dtype=torch.float32).unsqueeze(0)

MODEL_CPU.eval()
with torch.no_grad():
    preds = MODEL_CPU(X_t, tf_t)

pred_primary = preds[0, -1].item()
all_steps    = [round(float(preds[0, h]), 6) for h in range(HORIZON)]
direction    = 1 if pred_primary > 0 else 0
confidence   = 1.0 / (1.0 + math.exp(-abs(pred_primary) * 100))
signal, strength = generate_signal_v2(direction, confidence, pred_primary)
dir_label    = "UP ▲" if direction == 1 else "DOWN ▼"
agree        = all(s > 0 for s in all_steps) or all(s < 0 for s in all_steps)

latest_date = df_inf["datetime"].iloc[-1] if "datetime" in df_inf.columns else "N/A"

print("=" * 58)
print(f"  INFERENCE RESULT — {TEST_TICKER}")
print("=" * 58)
print(f"  Latest data:       {latest_date}")
print(f"  Signal:            {signal}  ({strength})")
print(f"  Direction:         {dir_label}")
print(f"  Confidence:        {confidence:.1%}")
print(f"  Primary ({HORIZON}d ret): {pred_primary:+.2%}")
print(f"  All horizon steps: {[f'{s:+.2%}' for s in all_steps]}")
print(f"  Step agreement:    {'✓ All agree' if agree else '~ Steps diverge'}")
print("=" * 58)
print()
print("This output matches what 'python infer.py --symbol RELIANCE' produces locally.")
print("step_agreement=True is a stronger signal (all horizon steps point same direction).")


## ✅ Complete — What to Do Next

### Files produced
| File | Size | Purpose |
|------|------|---------|
| `pretrained_v5.pth` | ~1 MB | Model weights — copy to `apps/ai-trading-service/` |
| `pretrained_v5_config.pth` | ~1 KB | Architecture config — needed to reload model |
| `scaler_v2.pkl` | ~10 KB | RobustScaler — normalises features at inference |
| `backtest_trades.csv` | varies | Full trade log with P&L |

### Copy to your local machine
```bash
# After downloading in Section 9, copy to:
apps/ai-trading-service/pretrained_v5.pth
apps/ai-trading-service/pretrained_v5_config.pth
apps/ai-trading-service/scaler_v2.pkl
```

### Local commands
```bash
# Daily inference (run after 3:45 PM IST)
python train_v5.py --mode finetune --symbol RELIANCE  # optional fine-tune

python infer.py --symbol RELIANCE
python infer.py --symbol RELIANCE --output json   # for n8n

# Backtest
python backtest_v2.py \
    --model pretrained_v5.pth \
    --config pretrained_v5_config.pth \
    --scaler scaler_v2.pkl \
    --data "data/RELIANCE/RELIANCE_daily_2010-01-01_2026-04-09.parquet" \
    --log_trades --csv trades.csv
```

### FastAPI (AWS EC2)
```bash
# Set environment variables
export MODEL_PATH=pretrained_v5.pth
export CONFIG_PATH=pretrained_v5_config.pth
export SCALER_PATH=scaler_v2.pkl

# Start server
uvicorn api_v2:app --host 0.0.0.0 --port 8000

# Test
curl -X POST http://localhost:8000/predict/upstox/auto \
  -H "Content-Type: application/json" \
  -d '{"candles": [[...Upstox candles...]]}'
```

### Expected accuracy benchmarks
| Training setup | Val accuracy | Notes |
|---|---|---|
| 4 correlated IT stocks | 52–55% | Marginal |
| 5–7 mixed sectors | 55–59% | Acceptable |
| 10 diverse sectors | 58–64% | Recommended |

### Required supporting files on local machine
No additional files needed beyond what's in `apps/ai-trading-service/`.
The 3 artifact files from this notebook replace the `model_v2.pth`, 
`model_v2_config.pth`, and `scaler_v2.pkl` that your existing pipeline uses.
